<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_03_feature_engineering/stage_03_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_03_feature_engineering**

# **0. Configuración del Entorno**


## 0.1. Clonado de repositorio / Acceso a Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


## 0.2. Instalación e importación de librerías


In [ ]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos

import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal



from scipy.stats import spearmanr

from scipy.stats import spearmanr
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd

In [ ]:
!{sys.executable} -m pip install -q ta
print("Librería instalada: technical-analysis")

  Preparing metadata (setup.py) ... done
Librería instalada: technical-analysis


In [ ]:
import ta
from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

## 0.4. Definición de rutas



In [ ]:
#/content/drive/MyDrive/neural_profit/data/processed/mnq_intraday_labeled.parquet
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

In [ ]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:
IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/targets/mnq_intraday_targets.parquet"))
#IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/stage_03b_target_definition_summary.json"))
OUT_PARQUET = Path(os.environ.get("OUT_PARQUET", "data/features/mnq_features_target.parquet"))
#OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/stage_04_feature_engineering_summary.json"))

In [ ]:
# PARA EL NOTEBOOK:
IN_PARQUET = DRIVE_DIR / IN_PARQUET
#IN_ARTIFACT = DRIVE_DIR / IN_ARTIFACT
OUT_PARQUET = DRIVE_DIR / OUT_PARQUET
#OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY

## 0.5. Códigos auxiliares para carga de datos y visualización


In [ ]:
def load_mnq_parquet():
    os.path.exists(IN_PARQUET)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(IN_PARQUET)
    return mnq_parquet

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import pandas as pd


def mnq_dataset_info(
    df: pd.DataFrame,
    *,
    name: str = "mnq_raw",
    tz_assume_if_naive: Optional[str] = None,  # ej: "UTC" o "America/New_York"
    day_def: str = "calendar",  # "calendar" (fecha calendario) o "trading" (días con datos)
) -> Dict[str, Any]:
    """
    Resume un dataset OHLCV con DatetimeIndex (ideal para mnq_raw).

    - Si el índice es tz-naive:
        - Si tz_assume_if_naive != None, lo localiza a esa tz.
        - Si no, reporta "tz-naive" (no se puede afirmar horario UTC).
    - Devuelve dict con métricas principales (y lo imprime bonito si se desea).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: se requiere DatetimeIndex, recibido: {type(df.index)}")

    idx = df.index

    # --- timezone / UTC info ---
    tzinfo = idx.tz
    if tzinfo is None:
        tz_status = "tz-naive (sin zona horaria)"
        if tz_assume_if_naive:
            idx = idx.tz_localize(tz_assume_if_naive)
            tzinfo = idx.tz
            tz_status = f"localizado como {tzinfo}"
    else:
        tz_status = f"{tzinfo}"

    # --- rango temporal ---
    ts_min = idx.min()
    ts_max = idx.max()

    first_day = ts_min.date()
    last_day = ts_max.date()

    # --- días ---
    if day_def == "calendar":
        total_days = (pd.Timestamp(last_day) - pd.Timestamp(first_day)).days + 1
    elif day_def == "trading":
        total_days = idx.normalize().nunique()
    else:
        raise ValueError("day_def debe ser 'calendar' o 'trading'")

    # --- columnas ---
    columns = list(df.columns)

    # --- checks útiles ---
    n_rows = len(df)
    n_cols = df.shape[1]
    n_missing = int(df.isna().sum().sum())
    missing_by_col = df.isna().sum().to_dict()
    dup_index = int(idx.duplicated().sum())
    is_monotonic = bool(idx.is_monotonic_increasing)

    # Frecuencia estimada (puede fallar si hay huecos grandes)
    freq = pd.infer_freq(idx[: min(50000, len(idx))])  # muestra grande pero acotada

    # Cobertura por día (min/max de hora del día, en tz del índice)
    # (Útil para ver si es 24/7 o horario de sesión)
    tod = pd.Series(idx.time)
    # Convertimos time a minutos del día para resumen robusto
    tod_minutes = pd.Series([t.hour * 60 + t.minute for t in tod])
    typical_minute_min = int(tod_minutes.min())
    typical_minute_max = int(tod_minutes.max())

    # Rango promedio de filas por día (sólo días con datos)
    rows_per_day = df.groupby(idx.normalize()).size()
    rows_per_day_stats = {
        "days_with_data": int(rows_per_day.shape[0]),
        "rows_per_day_min": int(rows_per_day.min()),
        "rows_per_day_p50": float(rows_per_day.median()),
        "rows_per_day_max": int(rows_per_day.max()),
    }

    # Si hay tz, también mostramos rango en UTC
    if tzinfo is not None:
        ts_min_utc = ts_min.tz_convert("UTC")
        ts_max_utc = ts_max.tz_convert("UTC")
        utc_range = (str(ts_min_utc), str(ts_max_utc))
        utc_note = "El índice está tz-aware; el horario UTC es inequívoco."
    else:
        utc_range = None
        utc_note = "El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones."

    info: Dict[str, Any] = {
        "name": name,
        "shape": (n_rows, n_cols),
        "columns": columns,
        "index_type": type(df.index).__name__,
        "index_tz": tz_status,
        "utc_note": utc_note,
        "datetime_min": str(ts_min),
        "datetime_max": str(ts_max),
        "first_day": str(first_day),
        "last_day": str(last_day),
        "total_days": int(total_days),
        "day_definition": day_def,
        "utc_range_if_applicable": utc_range,
        "is_index_monotonic_increasing": is_monotonic,
        "duplicated_timestamps_in_index": dup_index,
        "inferred_freq_sample": freq,
        "missing_total_cells": n_missing,
        "missing_by_col": missing_by_col,
        "rows_per_day_stats": rows_per_day_stats,
        "time_of_day_minutes_range": {
            "min_minute_of_day": typical_minute_min,
            "max_minute_of_day": typical_minute_max,
        },
    }
    return info


def print_mnq_dataset_info(info: Dict[str, Any]) -> None:
    """Imprime el dict de mnq_dataset_info de forma ordenada."""
    print(f"Dataset: {info['name']}")
    print(f"Shape: {info['shape']}")
    print(f"Columns: {info['columns']}")
    print(f"Index: {info['index_type']} | TZ: {info['index_tz']}")
    print(f"Datetime min/max: {info['datetime_min']}  ->  {info['datetime_max']}")
    print(f"First/Last day: {info['first_day']}  ->  {info['last_day']}")
    print(f"Total days ({info['day_definition']}): {info['total_days']}")
    #print(f"Inferred freq (sample): {info['inferred_freq_sample']}")
    #print(f"Index monotonic increasing: {info['is_index_monotonic_increasing']}")
    #print(f"Duplicated timestamps in index: {info['duplicated_timestamps_in_index']}")
    #print(f"Missing total cells: {info['missing_total_cells']}")
    #print(f"Missing by col: {info['missing_by_col']}")
    #print(f"Rows/day stats: {info['rows_per_day_stats']}")
    print(f"Time-of-day range (minutes): {info['time_of_day_minutes_range']}")
    print(f"UTC note: {info['utc_note']}")
    if info["utc_range_if_applicable"] is not None:
        print(f"UTC range: {info['utc_range_if_applicable'][0]}  ->  {info['utc_range_if_applicable'][1]}")

In [ ]:
mnq_intraday_targets = load_mnq_parquet()
info = mnq_dataset_info(mnq_intraday_targets, name="mnq_intraday", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info)

Archivo encontrado en disco. Cargando dataset local...
Dataset: mnq_intraday
Shape: (894845, 19)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_overnight', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri', 'delta_60', 'delta_90']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 04:30:00-05:00  ->  2025-06-13 16:00:00-04:00
First/Last day: 2020-01-02  ->  2025-06-13
Total days (trading): 1295
Time-of-day range (minutes): {'min_minute_of_day': 270, 'max_minute_of_day': 960}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 09:30:00+00:00  ->  2025-06-13 20:00:00+00:00


Gus, avancemos directo a lo importante: **convertir esto en un análisis útil y reusable**.

---

## 1. Función general (para cualquier régimen)

```python
def compute_sign_consistency(ic_table: pd.DataFrame) -> pd.DataFrame:
    df = ic_table.copy()

    df = df[
        df["IC_IS"].notna() &
        df["IC_OOS"].notna()
    ].copy()

    df["same_sign"] = (
        np.sign(df["IC_IS"]) == np.sign(df["IC_OOS"])
    )

    return df
```

---

## 2. Aplicarlo a todos los regímenes

```python
ic_premarket_sc = compute_sign_consistency(ic_table_premarket)
ic_opening_sc   = compute_sign_consistency(ic_table_opening)
ic_regular_sc   = compute_sign_consistency(ic_table_regular)
ic_closing_sc   = compute_sign_consistency(ic_table_closing)
ic_overnight_sc = compute_sign_consistency(ic_table_overnight)
```

---

## 3. Filtrar solo indicadores consistentes

```python
def get_consistent_indicators(df):
    return df[df["same_sign"]].copy()

cons_premarket = get_consistent_indicators(ic_premarket_sc)
cons_opening   = get_consistent_indicators(ic_opening_sc)
cons_regular   = get_consistent_indicators(ic_regular_sc)
cons_closing   = get_consistent_indicators(ic_closing_sc)
cons_overnight = get_consistent_indicators(ic_overnight_sc)
```

---

## 4. Ranking solo con consistentes (CLAVE)

```python
def rank_consistent(df, top_n=10):
    return (
        df.sort_values("abs_IC_OOS", ascending=False)
          .head(top_n)
    )

top_cons_premarket = rank_consistent(cons_premarket)
top_cons_opening   = rank_consistent(cons_opening)
top_cons_regular   = rank_consistent(cons_regular)
top_cons_closing   = rank_consistent(cons_closing)
top_cons_overnight = rank_consistent(cons_overnight)
```

---

## 5. Métrica resumen por régimen (muy útil)

```python
def summarize_sign_consistency(df):
    return {
        "n_total": len(df),
        "n_consistent": df["same_sign"].sum(),
        "pct_consistent": df["same_sign"].mean()
    }

summary_sign = pd.DataFrame({
    "premarket": summarize_sign_consistency(ic_premarket_sc),
    "opening": summarize_sign_consistency(ic_opening_sc),
    "regular": summarize_sign_consistency(ic_regular_sc),
    "closing": summarize_sign_consistency(ic_closing_sc),
    "overnight": summarize_sign_consistency(ic_overnight_sc),
}).T
```

---

## 6. Interpretación (muy puntual)

Qué estás validando:

* Un factor debe mantener **dirección económica estable**
* Si cambia de signo:

  * es ruido
  * o depende del régimen
  * o está sobreajustado

Esto es crítico porque:

> Un alpha factor debe ser **estable en dirección**, no solo en magnitud

---

## 7. Regla práctica (para stage siguiente)

Qué deberías quedarte:

* ✔ mismo signo
* ✔ alto `abs_IC_OOS`
* ✔ gap bajo (lo hicimos antes)

Qué deberías descartar:

* ✘ cambia de signo
* ✘ IC alto en IS pero distinto signo en OOS

---

## 8. Output que deberías guardar

Para cada régimen:

* tabla completa con `same_sign`
* top consistentes
* resumen (% consistencia)

---

Si quieres, el siguiente paso natural es:
👉 cruzar **consistencia + robustez + ranking** para construir el **feature set final automáticamente**.


## 0.6. Auxiliares

In [ ]:
def assign_indicator_family(indicator: str) -> str:
    """
    Asigna una familia económica a cada indicador técnico
    en función de su nombre.
    """
    name = indicator.lower()

    if "ema" in name or name.startswith("price_"):
        return "trend_price"

    if name.startswith("bb_"):
        return "volatility_extension"

    if name.startswith("roc") or name.startswith("momentum"):
        return "momentum"

    if name.startswith("rsi"):
        return "momentum_oscillator"

    if name.startswith("stoch"):
        return "momentum_oscillator"

    if name.startswith("atr"):
        return "volatility"

    if name.startswith("volume_ratio"):
        return "volume"

    if name == "macd":
        return "trend_momentum"

    return "other"

In [ ]:
def select_top_by_family(
    ic_df: pd.DataFrame,
    top_n: int = 1
) -> pd.DataFrame:
    """
    Selecciona los mejores indicadores por familia
    según abs_IC_delta.
    """
    df = ic_df.copy()

    # Asignar familia
    df["family"] = df["indicator"].apply(assign_indicator_family)

    # Ordenar por fuerza de señal
    df = df.sort_values("abs_IC_delta", ascending=False)

    # Tomar top N por familia
    df_top = (
        df.groupby("family", as_index=False)
          .head(top_n)
          .reset_index(drop=True)
    )

    return df_top

# **Introducción**

El objetivo de este stage es investigar y construir un conjunto de indicadores técnicos y transformaciones derivadas que puedan actuar como variables explicativas para los targets previamente definidos (`delta_60` y `delta_90`).

Una vez definido el problema de predicción, el proceso de machine learning aplicado a mercados financieros continúa con la etapa de **feature engineering**, cuyo propósito es transformar los datos crudos de mercado (OHLCV e información temporal) en señales informativas que capturen patrones explotables. En este contexto, los indicadores técnicos no se consideran como reglas de trading directas, sino como **aproximaciones cuantitativas de comportamientos del mercado**, tales como tendencia, momentum, volatilidad, reversión a la media y estructura intradía.

Este stage se centra en diseñar, calcular y evaluar un conjunto amplio pero estructurado de features candidatas, incluyendo:

* Indicadores técnicos clásicos (momentum, ROC, EMA, ATR, etc.)
* Versiones parametrizadas de dichos indicadores (distintas ventanas)
* Transformaciones normalizadas (por volatilidad, rango, etc.)
* Interacciones entre variables (combinaciones no lineales)
* Features condicionadas por régimen de mercado (horario, día de la semana, sesión)

El objetivo no es únicamente generar features, sino **evaluar rigurosamente su capacidad de aportar señal predictiva** respecto a los targets definidos. Para ello, cada feature será analizada en términos de:

* Relación estadística con el target
* Estabilidad temporal
* Comportamiento bajo distintos regímenes de mercado
* Robustez frente a ruido

Como resultado de este stage, se obtendrá un conjunto depurado de variables explicativas (alpha factors) que servirán como base para el entrenamiento de modelos en etapas posteriores. Este proceso es crítico, ya que la calidad de las features determina en gran medida el desempeño final de cualquier modelo predictivo en el contexto financiero.

# **Plan de investigación de indicadores técnicos (Alpha Factors)**


Este stage tiene como objetivo evaluar la capacidad predictiva de distintos indicadores técnicos sobre targets intradía (delta_60 y delta_90), siguiendo un enfoque sistemático alineado con el workflow de machine learning para trading.

El proceso se estructura en las siguientes etapas:

---

1. Análisis del dataset de entrada y targets
   
    Se analiza el dataset base que ya contiene:

    * variables OHLCV
    * flags de régimen de mercado
    * targets definidos (delta_60, delta_90)

    Se verifican:

    * orden temporal correcto
    * ausencia de leakage
    * consistencia de los targets
    * distribución de los targets

    Objetivo:
    - Validar que el problema esté bien definido antes de construir features.

---

2. Construcción de indicadores técnicos

   Se generan múltiples indicadores a partir de OHLCV, incluyendo:

    * EMA
    * ROC
    * RSI
    * Bollinger Bands
    * STOCH
    * ATR
    * otros derivados

    utilizando distintas ventanas temporales.

    Objetivo:
    Construir un conjunto amplio de candidatos a factores predictivos.

---

3. Separación IS vs OOS
   El dataset se divide en:

* In-Sample (IS): exploración
* Out-of-Sample (OOS): validación

Objetivo:
Evitar overfitting y evaluar generalización.

---

4. Evaluación mediante Information Coefficient (IC)
   
   Para cada indicador, horizonte y régimen:

    * IC_IS
    * IC_OOS
    * abs_IC_OOS
    * gap IS vs OOS

    Objetivo:
    Medir la relación entre cada indicador y el target futuro.

---

5. Análisis de robustez IS vs OOS
   
   Se evalúa:

* diferencia entre IC_IS e IC_OOS
* magnitud del gap

Interpretación:

* gap bajo → robusto
* gap alto → inestable

Objetivo:
Detectar señales que generalizan.

---

6. Consistencia de signo
   
   Se calcula:

* same_sign = sign(IC_IS) == sign(IC_OOS)

Interpretación:

* True → señal consistente
* False → señal no confiable

Objetivo:
Eliminar señales inestables.

---

7. Análisis por régimen de mercado
   
   Se evalúan los indicadores en:

* overnight
* premarket
* opening
* regular
* closing

Objetivo:

* detectar factores estructurales
* detectar factores contextuales

---

8. Ranking de indicadores
   
   Se priorizan indicadores según:

* magnitud de IC_OOS
* estabilidad (gap bajo)
* consistencia de signo

Objetivo:
Identificar los factores más relevantes.

---

9. Análisis de repetición entre regímenes
   
   Se identifican indicadores que:

* aparecen consistentemente en varios regímenes

Objetivo:
Detectar factores robustos globales.

---

10. Identificación de indicadores específicos por régimen
    
    Se buscan factores que:

* funcionan bien en un régimen
* no en otros

Objetivo:
Capturar señales contextuales.

---

11. Análisis de redundancia (correlación entre indicadores)

Se evalúa:

* correlación entre features
* agrupación de indicadores similares

Objetivo:

* eliminar duplicados
* reducir dimensionalidad
* evitar multicolinealidad

---

12. Selección final de features

Se seleccionan indicadores según:

* señal OOS
* robustez
* consistencia
* baja redundancia

Objetivo:
Definir el feature set final.

---

13. Interpretación económica de los factores

Se analiza qué tipo de señal capturan:

* momentum
* mean reversion
* volatilidad

Objetivo:
Validar coherencia con el mercado.

---

14. Output del stage

Se generan:

* tablas resumen
* rankings
* análisis por régimen
* features finales
* conclusiones

Objetivo final:
Determinar si existe señal predictiva y definir un conjunto robusto de features para el modelado.


# **1. Análisis del dataset de entrada y targets**

In [17]:
import numpy as np
import pandas as pd

def analyze_input_dataset_stage03(
    df: pd.DataFrame,
    *,
    datetime_index_required: bool = True,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    target_cols: tuple[str, ...] = ("delta_60", "delta_90"),
    ohlcv_cols: tuple[str, ...] = ("open", "high", "low", "close", "volume"),
    regime_flags: tuple[str, ...] = (
        "is_overnight",
        "is_premarket",
        "is_opening",
        "is_regular",
        "is_closing",
    ),
    print_report: bool = True,
) -> dict:
    """
    Analiza el dataset base del stage_03 antes de construir indicadores técnicos.

    Verifica:
    - presencia de columnas esperadas
    - orden temporal correcto
    - consistencia básica OHLCV
    - consistencia de flags de régimen
    - consistencia temporal de los targets
    - ausencia básica de leakage
    - distribución de targets

    Devuelve un diccionario con:
    - report: resumen general
    - target_summary: resumen estadístico de targets
    - regime_summary: muestras por régimen
    - leakage_rows: filas sospechosas de leakage temporal
    - invalid_ohlc_rows: filas con OHLC inconsistente
    - invalid_regime_rows: filas con flags de régimen inconsistentes
    """

    data = df.copy()

    # ============================================================
    # 0. Validaciones de columnas
    # ============================================================
    required_cols = set(ohlcv_cols) | set(regime_flags) | set(target_cols) | {date_col, minute_col}
    missing_cols = sorted([c for c in required_cols if c not in data.columns])

    if missing_cols:
        raise ValueError(f"Faltan columnas requeridas: {missing_cols}")

    # ============================================================
    # 1. Validación / normalización temporal
    # ============================================================
    if datetime_index_required and not isinstance(data.index, pd.DatetimeIndex):
        raise TypeError("El DataFrame debe tener DatetimeIndex.")

    data[date_col] = pd.to_datetime(data[date_col])

    if isinstance(data.index, pd.DatetimeIndex):
        data = data.sort_index().copy()
        is_sorted = data.index.is_monotonic_increasing
    else:
        data = data.sort_values([date_col, minute_col]).copy()
        is_sorted = (
            data[[date_col, minute_col]]
            .reset_index(drop=True)
            .equals(
                data[[date_col, minute_col]]
                .sort_values([date_col, minute_col])
                .reset_index(drop=True)
            )
        )

    # ============================================================
    # 2. Consistencia OHLC
    # ============================================================
    invalid_ohlc_mask = (
        (data["high"] < data["low"]) |
        (data["open"] > data["high"]) |
        (data["open"] < data["low"]) |
        (data["close"] > data["high"]) |
        (data["close"] < data["low"]) |
        (data["volume"] < 0)
    )
    invalid_ohlc_rows = data.loc[invalid_ohlc_mask].copy()

    # ============================================================
    # 3. Consistencia de flags de régimen
    #    Esperamos exactamente 1 régimen activo por fila
    # ============================================================
    regime_sum = data[list(regime_flags)].fillna(0).astype(int).sum(axis=1)
    invalid_regime_mask = regime_sum != 1
    invalid_regime_rows = data.loc[invalid_regime_mask].copy()

    regime_summary = pd.DataFrame({
        "regime_flag": regime_flags,
        "n_rows": [int(data[r].fillna(0).astype(int).sum()) for r in regime_flags],
    })

    # ============================================================
    # 4. Ausencia básica de leakage temporal
    #    Validamos que el target exista solo si hay futuro suficiente
    # ============================================================
    if not isinstance(data.index, pd.DatetimeIndex):
        raise TypeError("Para validar leakage temporal, el DataFrame debe tener DatetimeIndex.")

    # próximo timestamp por fila
    data["_next_ts"] = data.index.to_series().shift(-1)
    data["_next_date"] = data[date_col].shift(-1)

    leakage_checks = []

    for target_col in target_cols:
        horizon = int(target_col.split("_")[1])

        # timestamp futuro esperado dentro del mismo día
        expected_future_ts = data.index + pd.Timedelta(minutes=horizon)

        # último timestamp por día
        last_ts_by_day = data.groupby(date_col).apply(lambda g: g.index.max())
        last_ts_by_day.name = "last_ts_day"

        data = data.merge(
            last_ts_by_day,
            left_on=date_col,
            right_index=True,
            how="left"
        )

        # si expected_future_ts supera el último ts del día,
        # el target debería ser NaN
        invalid_future_mask = expected_future_ts > data["last_ts_day"]

        suspect_non_nan_target_mask = invalid_future_mask & data[target_col].notna()

        leakage_checks.append(
            data.loc[suspect_non_nan_target_mask, [date_col, minute_col, target_col]].assign(
                target_col_name=target_col
            )
        )

        data = data.drop(columns=["last_ts_day"])

    leakage_rows = pd.concat(leakage_checks, axis=0) if leakage_checks else pd.DataFrame()

    # ============================================================
    # 5. Consistencia de targets
    # ============================================================
    target_summary_rows = []

    for target_col in target_cols:
        s = data[target_col]

        target_summary_rows.append({
            "target_col": target_col,
            "n_total": int(len(s)),
            "n_notna": int(s.notna().sum()),
            "n_nan": int(s.isna().sum()),
            "pct_nan": float(s.isna().mean()),
            "mean": float(s.mean()) if s.notna().any() else np.nan,
            "std": float(s.std()) if s.notna().any() else np.nan,
            "min": float(s.min()) if s.notna().any() else np.nan,
            "p01": float(s.quantile(0.01)) if s.notna().any() else np.nan,
            "p05": float(s.quantile(0.05)) if s.notna().any() else np.nan,
            "median": float(s.median()) if s.notna().any() else np.nan,
            "p95": float(s.quantile(0.95)) if s.notna().any() else np.nan,
            "p99": float(s.quantile(0.99)) if s.notna().any() else np.nan,
            "max": float(s.max()) if s.notna().any() else np.nan,
            "pct_positive": float((s > 0).mean()) if s.notna().any() else np.nan,
            "pct_negative": float((s < 0).mean()) if s.notna().any() else np.nan,
            "pct_zero": float((s == 0).mean()) if s.notna().any() else np.nan,
        })

    target_summary = pd.DataFrame(target_summary_rows)

    # ============================================================
    # 6. Verificación de duplicados temporales
    # ============================================================
    if isinstance(data.index, pd.DatetimeIndex):
        dup_time_count = int(data.index.duplicated().sum())
    else:
        dup_time_count = int(
            data.duplicated(subset=[date_col, minute_col]).sum()
        )

    # ============================================================
    # 7. Resumen general
    # ============================================================
    report = {
        "n_rows": int(len(data)),
        "n_cols": int(data.shape[1]),
        "datetime_index_type": str(type(data.index)),
        "is_sorted_temporally": bool(is_sorted),
        "n_unique_days": int(data[date_col].nunique()),
        "datetime_duplicates": dup_time_count,
        "n_invalid_ohlc_rows": int(len(invalid_ohlc_rows)),
        "n_invalid_regime_rows": int(len(invalid_regime_rows)),
        "n_suspect_leakage_rows": int(len(leakage_rows)),
    }

    # ============================================================
    # 8. Print reporte
    # ============================================================
    if print_report:
        print("=" * 90)
        print("STAGE_03 | ANÁLISIS DEL DATASET DE ENTRADA")
        print("=" * 90)
        print(f"Filas: {report['n_rows']}")
        print(f"Columnas: {report['n_cols']}")
        print(f"Días únicos: {report['n_unique_days']}")
        print(f"Orden temporal correcto: {report['is_sorted_temporally']}")
        print(f"Timestamps duplicados: {report['datetime_duplicates']}")
        print(f"Filas con OHLC inconsistente: {report['n_invalid_ohlc_rows']}")
        print(f"Filas con régimen inconsistente: {report['n_invalid_regime_rows']}")
        print(f"Filas sospechosas de leakage temporal: {report['n_suspect_leakage_rows']}")
        print()

        print("-" * 90)
        print("MUESTRAS POR RÉGIMEN")
        print("-" * 90)
        print(regime_summary.to_string(index=False))
        print()

        print("-" * 90)
        print("RESUMEN DE TARGETS")
        print("-" * 90)
        print(target_summary.to_string(index=False))
        print()

        if len(invalid_ohlc_rows) > 0:
            print("-" * 90)
            print("MUESTRA DE FILAS CON OHLC INVÁLIDO")
            print("-" * 90)
            print(invalid_ohlc_rows.head(10).to_string())

        if len(invalid_regime_rows) > 0:
            print("-" * 90)
            print("MUESTRA DE FILAS CON FLAGS DE RÉGIMEN INVÁLIDOS")
            print("-" * 90)
            print(invalid_regime_rows.head(10).to_string())

        if len(leakage_rows) > 0:
            print("-" * 90)
            print("MUESTRA DE FILAS SOSPECHOSAS DE LEAKAGE")
            print("-" * 90)
            print(leakage_rows.head(10).to_string(index=False))

    # limpieza auxiliares
    drop_aux = [c for c in ["_next_ts", "_next_date"] if c in data.columns]
    if drop_aux:
        data = data.drop(columns=drop_aux)

    return {
        "report": report,
        "target_summary": target_summary,
        "regime_summary": regime_summary,
        "invalid_ohlc_rows": invalid_ohlc_rows,
        "invalid_regime_rows": invalid_regime_rows,
        "leakage_rows": leakage_rows,
    }

In [18]:
stage03_input_report = analyze_input_dataset_stage03(
    mnq_intraday_targets,
    print_report=True,
)

STAGE_03 | ANÁLISIS DEL DATASET DE ENTRADA
Filas: 894845
Columnas: 21
Días únicos: 1295
Orden temporal correcto: True
Timestamps duplicados: 0
Filas con OHLC inconsistente: 0
Filas con régimen inconsistente: 0
Filas sospechosas de leakage temporal: 0

------------------------------------------------------------------------------------------
MUESTRAS POR RÉGIMEN
------------------------------------------------------------------------------------------
 regime_flag  n_rows
is_overnight  312095
is_premarket   77700
  is_opening   77700
  is_regular  388500
  is_closing   38850

------------------------------------------------------------------------------------------
RESUMEN DE TARGETS
------------------------------------------------------------------------------------------
target_col  n_total  n_notna  n_nan  pct_nan     mean       std      min    p01     p05  median   p95    p99    max  pct_positive  pct_negative  pct_zero
  delta_60   894845   817145  77700 0.086831 0.480395 55.146253

# **2. Indicadores Técnicos**

Los indicadores técnicos calculados en cada jornada tienen como objetivo capturar dinámicas intradía relevantes del precio y el volumen, tales como momentum, sobrecompra/sobreventa, presión institucional o posibles reversiones. Cada uno aporta información complementaria sobre el comportamiento del mercado a corto plazo. En particular:

## **2.1. Indicadores técnicos individuales**

#### 1. **RSI (Relative Strength Index)**

Mide la fuerza relativa del precio en los últimos períodos (3, 5, 7, 14), oscilando entre 0 y 100.

  - Valores altos indican posibles condiciones de sobrecompra, mientras que valores bajos sugieren sobreventa.
  
  - Calculado sobre los precios de cierre intradía, el RSI es útil para identificar puntos de reversión potenciales en el corto plazo.

In [19]:
def calcular_rsi(df=mnq_intraday_targets, target='close' ):
  rsi_columns = ['rsi_14', 'rsi_7', 'rsi_5', 'rsi_3']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['rsi_14'] = ta.momentum.RSIIndicator(grupo[target], window=14).rsi()
        grupo['rsi_7'] = ta.momentum.RSIIndicator(grupo[target], window=7).rsi()
        grupo['rsi_5'] = ta.momentum.RSIIndicator(grupo[target], window=5).rsi()
        grupo['rsi_3'] = ta.momentum.RSIIndicator(grupo[target], window=3).rsi()
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)
  return df, rsi_columns

#### 2. **Momentum**

Mide la aceleración reciente del precio mediante la variación porcentual entre el precio actual y el de hace N minutos.

  - Un valor positivo indica una subida reciente, lo que podría sugerir una continuación alcista.

  - Un valor negativo señala presión bajista reciente, potencialmente anticipando una continuación a la baja.

In [20]:
def calcular_momentum(df=mnq_intraday_targets, target='close' ):
  momentum_columns = ['mom_10', 'mom_5','mom_3']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['mom_10'] = grupo[target].pct_change(10)
        grupo['mom_5'] = grupo[target].pct_change(5)
        grupo['mom_3'] = grupo[target].pct_change(3)
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)
  return df, momentum_columns

#### 3. **Relación de volumen actual vs. su promedio reciente**

Compara el volumen actual con su media móvil en distintas ventanas de tiempo: 15, 20 y 30 minutos.

  - Un valor mayor a 1 indica un volumen superior al promedio de la ventana correspondiente, lo que puede reflejar interés creciente o actividad institucional.

  - Un valor menor a 1 sugiere baja actividad o consolidación del precio.

Esta métrica permite detectar aumentos de volumen ("spikes") sin depender del volumen en crudo, y las diferentes ventanas permiten capturar variaciones en la dinámica de corto plazo con distinta sensibilidad.


In [21]:
def calcular_volumen_ratio(df=mnq_intraday_targets, target='close'):
  volume_ratio_columns = ['volume_ratio_15', 'volume_ratio_20', 'volume_ratio_30', 'volume_ratio_60', 'volume_ratio_90']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['volume_ratio_15'] = grupo['volume'] / grupo['volume'].rolling(15).mean()
        grupo['volume_ratio_20'] = grupo['volume'] / grupo['volume'].rolling(20).mean()
        grupo['volume_ratio_30'] = grupo['volume'] / grupo['volume'].rolling(30).mean()
        grupo['volume_ratio_60'] = grupo['volume'] / grupo['volume'].rolling(60).mean()
        grupo['volume_ratio_90'] = grupo['volume'] / grupo['volume'].rolling(90).mean()
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

  return df, volume_ratio_columns

#### 4. **MACD diferencial (señal de cruce)**

Representa la diferencia entre la línea MACD y su línea de señal (una media exponencial de sí misma).

  - Un valor positivo y creciente indica momentum alcista.

  - Un valor negativo sugiere presión bajista.
  
Es ampliamente utilizado para detectar giros de tendencia y cambios en la dinámica del mercado.


In [22]:
def calcular_macd(df=mnq_intraday_targets, target='close'):
    macd_columns = ['macd']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['macd'] = ta.trend.MACD(grupo[target]).macd_diff()
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, macd_columns

#### 5. **Distancia del precio actual a su EMA (15, 20 y 30 minutos)**

Mide el desvío porcentual del precio respecto a su media exponencial en diferentes ventanas, y actúa como indicador de sobreextensión o retorno a la media.

  - Si el precio está muy por encima de la EMA, puede anticipar una reversión bajista o una posible aceleración alcista.

  - Si está por debajo, podría indicar agotamiento o presión vendedora.<br>

Esta métrica se expresa como un porcentaje relativo, lo que facilita la comparación entre distintas ventanas temporales y condiciones de mercado.

Usar varias ventanas (15, 20 y 30 minutos) permite capturar diferentes horizontes de reacción del precio frente a su media móvil.


In [23]:
def calcular_ema(df=mnq_intraday_targets, target='close'):
    ema_columns = ['ema_15', 'ema_20', 'ema_30',  'ema_60']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['ema_15'] = grupo[target] / grupo[target].ewm(span=15).mean() - 1
        grupo['ema_20'] = grupo[target] / grupo[target].ewm(span=20).mean() - 1
        grupo['ema_30'] = grupo[target] / grupo[target].ewm(span=30).mean() - 1
        grupo['ema_60'] = grupo[target] / grupo[target].ewm(span=60).mean() - 1
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, ema_columns

#### 6. **%K Estocástico**

Mide la posición relativa del precio actual dentro del rango alto-bajo de los últimos n periodos (generalmente 14).

  - Se utiliza para identificar condiciones extremas de sobrecompra o sobreventa.

  - Un valor cercano a 100 indica que el precio está cerca del máximo reciente (potencial sobrecompra), mientras que un valor cercano a 0 indica proximidad al mínimo reciente (posible sobreventa).

Es útil para detectar momentos en los que el precio puede estar excesivamente extendido y susceptible a una reversión.


In [24]:
def calcular_stochastic(df=mnq_intraday_targets, target='close'):
    stoch_columns = ['stoch_k_14', 'stoch_k_20', 'stoch_k_30']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        stoch_14 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=14, smooth_window=3
        )
        grupo['stoch_k_14'] = stoch_14.stoch()

        stoch_20 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=20, smooth_window=3
        )
        grupo['stoch_k_20'] = stoch_20.stoch()

        stoch_30 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=30, smooth_window=3
        )
        grupo['stoch_k_30'] = stoch_30.stoch()

        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, stoch_columns


#### 7.**%B de Bollinger (Bollinger Band Percent)**

Indica la posición del precio actual en relación con las bandas de Bollinger, que están construidas alrededor de una media móvil usando desviaciones estándar.

  - Un valor de %B > 1 sugiere que el precio está por encima de la banda superior, lo que podría implicar exceso de optimismo o momentum fuerte.

  - Un valor < 0 indica que está por debajo de la banda inferior, posible señal de pánico o sobreventa extrema.

Este indicador es eficaz para identificar zonas de congestión, breakout o reversiones basadas en la volatilidad reciente.


In [25]:
from ta.volatility import BollingerBands

def calcular_bollinger(df=mnq_intraday_targets, target='close'):
    '''bollinger_columns = [
        'bb_percent_15_15', 'bb_percent_20_15', 'bb_percent_30_15',
        'bb_percent_15_20', 'bb_percent_20_20', 'bb_percent_30_20',
        'bb_percent_15_25', 'bb_percent_20_25', 'bb_percent_30_25',
    ]'''

    bollinger_columns = [
        'bb_15_15', 'bb_20_15', 'bb_30_15', 'bb_60_15',
        'bb_15_20', 'bb_20_20', 'bb_30_20', 'bb_60_20',
        'bb_15_25', 'bb_20_25', 'bb_30_25', 'bb_60_25',
    ]

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        # std: 1.5
        grupo['bb_15_15'] = BollingerBands(grupo[target], window=15, window_dev=1.5).bollinger_pband()
        grupo['bb_20_15'] = BollingerBands(grupo[target], window=20, window_dev=1.5).bollinger_pband()
        grupo['bb_30_15'] = BollingerBands(grupo[target], window=30, window_dev=1.5).bollinger_pband()
        grupo['bb_60_15'] = BollingerBands(grupo[target], window=60, window_dev=1.5).bollinger_pband()

        # std: 2
        grupo['bb_15_20'] = BollingerBands(grupo[target], window=15, window_dev=2).bollinger_pband()
        grupo['bb_20_20'] = BollingerBands(grupo[target], window=20, window_dev=2).bollinger_pband()
        grupo['bb_30_20'] = BollingerBands(grupo[target], window=30, window_dev=2).bollinger_pband()
        grupo['bb_60_20'] = BollingerBands(grupo[target], window=60, window_dev=2).bollinger_pband()

        # std: 2.5
        grupo['bb_15_25'] = BollingerBands(grupo[target], window=15, window_dev=2.5).bollinger_pband()
        grupo['bb_20_25'] = BollingerBands(grupo[target], window=20, window_dev=2.5).bollinger_pband()
        grupo['bb_30_25'] = BollingerBands(grupo[target], window=30, window_dev=2.5).bollinger_pband()
        grupo['bb_60_25'] = BollingerBands(grupo[target], window=60, window_dev=2.5).bollinger_pband()



        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, bollinger_columns


In [26]:
def calcular_bollinger_resume(df=mnq_intraday_targets, target='close'):

    bollinger_columns = [
        'bb_15', 'bb_20', 'bb_30', 'bb_60',
    ]

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        # std: 2
        grupo['bb_15'] = BollingerBands(grupo[target], window=15, window_dev=2).bollinger_pband()
        grupo['bb_20'] = BollingerBands(grupo[target], window=20, window_dev=2).bollinger_pband()
        grupo['bb_30'] = BollingerBands(grupo[target], window=30, window_dev=2).bollinger_pband()
        grupo['bb_60'] = BollingerBands(grupo[target], window=60, window_dev=2).bollinger_pband()
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, bollinger_columns

#### 8. **ATR normalizado (Average True Range / precio)**

Representa la volatilidad absoluta reciente ajustada al nivel del precio.

  - El ATR mide el rango promedio de oscilación de un activo en los últimos n periodos, capturando tanto movimientos bruscos como gaps.

  - Al normalizarlo dividiéndolo por el precio, se obtiene una medida relativa, comparable entre distintos niveles de mercado.

Este indicador es útil para detectar momentos de alta o baja volatilidad intradía, que pueden influir en la confiabilidad de otras señales técnicas.


In [27]:
def calcular_atr(df=mnq_intraday_targets, target='close', windows=[5, 10, 14, 20, 30]):
    atr_columns = []

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        for w in windows:
            col_name = f'atr_norm_{w}'
            atr = AverageTrueRange(high=grupo['high'], low=grupo['low'], close=grupo[target], window=w)
            grupo[col_name] = atr.average_true_range() / grupo[target]
            atr_columns.append(col_name)
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, list(set(atr_columns))

####  9. **ROC (Rate of Change)**

Calcula la tasa de cambio porcentual del precio con respecto a su valor n minutos atrás.

- Es un indicador de momentum que capta aceleraciones o desaceleraciones recientes del precio.

- Valores positivos indican presión alcista; negativos, presión bajista.

A diferencia del momentum tradicional, el ROC expresa el cambio de forma normalizada y en porcentaje, lo que facilita su interpretación comparativa entre distintos activos o marcos temporales.


In [28]:
def calcular_roc(df=mnq_intraday_targets, target='close'):
    roc_columns = ['roc_5', 'roc_10', 'roc_20','roc_30','roc_60']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['roc_5'] = ROCIndicator(close=grupo[target], window=5).roc()
        grupo['roc_10'] = ROCIndicator(close=grupo[target], window=10).roc()
        grupo['roc_20'] = ROCIndicator(close=grupo[target], window=20).roc()
        grupo['roc_30'] = ROCIndicator(close=grupo[target], window=30).roc()
        grupo['roc_60'] = ROCIndicator(close=grupo[target], window=60).roc()

        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, roc_columns

## **2.2. Cálculo de indicadores técnicos**

Calculamos los indicadores técnicos

In [36]:
import os
import glob
import numpy as np
import pandas as pd


def load_or_build_mnq_intraday_with_indicators(
    mnq_intraday_targets: pd.DataFrame,
    *,
    processed_dir: str = "/content/drive/MyDrive/neural_profit/data/targets",
    parquet_name: str = "mnq_intraday_with_indicators.parquet",
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    regime_flags: tuple[str, ...] = (
        "is_overnight",
        "is_premarket",
        "is_opening",
        "is_regular",
        "is_closing",
    ),
    dropna_after_indicators: bool = True,
    print_report: bool = True,
):
    """
    Carga un parquet existente con indicadores o los calcula desde mnq_intraday_targets.

    Además:
    - elimina NaNs al final del cálculo
    - reconstruye indicator_columns
    - valida orden cronológico
    - valida NaNs remanentes en indicadores
    - valida consistencia básica por día para indicadores rolling
    - imprime un reporte final del dataset
    """

    processed_dir = os.path.abspath(processed_dir)
    parquet_path = os.path.join(processed_dir, parquet_name)
    os.makedirs(processed_dir, exist_ok=True)

    # ============================================================
    # 1) Cargar o calcular
    # ============================================================
    loaded_from = None

    if os.path.exists(parquet_path):
        df = pd.read_parquet(parquet_path)
        loaded_from = parquet_path
        if print_report:
            print(f"[OK] Cargado: {parquet_path}")

    else:
        candidates = sorted(glob.glob(os.path.join(processed_dir, "*with_indicators*.parquet")))
        if candidates:
            df = pd.read_parquet(candidates[-1])
            loaded_from = candidates[-1]
            if print_report:
                print(f"[OK] Cargado (fallback): {candidates[-1]}")
        else:
            df = mnq_intraday_targets.copy()
            loaded_from = "computed"

            if print_report:
                print("[OK] Calculando indicadores técnicos...")

            df, rsi_columns = calcular_rsi(df)
            df, momentum_columns = calcular_momentum(df)
            df, volume_ratio_columns = calcular_volumen_ratio(df)
            df, macd_columns = calcular_macd(df)
            df, ema_columns = calcular_ema(df)
            df, stoch_columns = calcular_stochastic(df)
            df, bollinger_columns = calcular_bollinger(df)
            df, atr_columns = calcular_atr(df)
            df, roc_columns = calcular_roc(df)

            indicator_columns = (
                rsi_columns
                + momentum_columns
                + volume_ratio_columns
                + macd_columns
                + ema_columns
                + stoch_columns
                + bollinger_columns
                + atr_columns
                + roc_columns
            )

            # eliminar duplicados preservando orden
            indicator_columns = list(dict.fromkeys(indicator_columns))

            # ============================================================
            # 2) Eliminar NaNs SOLO al final
            # ============================================================
            if dropna_after_indicators:
                n_before_dropna = len(df)
                df = df.dropna(subset=indicator_columns).copy()
                n_after_dropna = len(df)
            else:
                n_before_dropna = len(df)
                n_after_dropna = len(df)

            # guardar parquet final
            df.to_parquet(parquet_path, index=True)
            if print_report:
                print(f"[OK] Calculado y guardado: {parquet_path}")

    # ============================================================
    # 3) Normalización temporal
    # ============================================================
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col])

    df = df.sort_index().copy()

    # ============================================================
    # 4) Reconstruir indicator_columns desde el dataset final
    # ============================================================
    columns_to_remove = {
        date_col,
        minute_col,
        "open", "high", "low", "close", "volume",
        "ret_60", "ret_90",
        "delta_60", "delta_90",
        "split_fe",
        *regime_flags,
        "is_mon", "is_tue", "is_wed", "is_thu", "is_fri",
    }

    indicator_columns = [col for col in df.columns if col not in columns_to_remove]

    # ============================================================
    # 5) Validaciones
    # ============================================================
    # 5.1 Orden cronológico
    is_sorted = df.index.is_monotonic_increasing

    # 5.2 NaNs en indicadores
    nan_counts_indicators = df[indicator_columns].isna().sum().sort_values(ascending=False)
    indicators_with_nan = nan_counts_indicators[nan_counts_indicators > 0]

    # 5.3 Filas por régimen
    regime_summary = pd.DataFrame({
        "regime_flag": regime_flags,
        "n_rows": [int(df[r].fillna(0).astype(int).sum()) if r in df.columns else 0 for r in regime_flags],
    })

    # 5.4 Validación básica: un solo régimen activo por fila
    existing_regimes = [r for r in regime_flags if r in df.columns]
    if existing_regimes:
        regime_sum = df[existing_regimes].fillna(0).astype(int).sum(axis=1)
        invalid_regime_rows = int((regime_sum != 1).sum())
    else:
        invalid_regime_rows = np.nan

    # 5.5 Validación básica: no cruce de días en indicadores
    # Idea:
    # para cada día, si tras dropna final aún existe al menos una fila, entonces
    # el primer minuto superviviente debería ser suficientemente tardío respecto
    # a ventanas rolling. No demuestra todo, pero detecta mezclas raras.
    day_minute_summary = (
        df.groupby(date_col)[minute_col]
        .agg(first_minute="min", last_minute="max", n_rows="count")
        .reset_index()
        if date_col in df.columns and minute_col in df.columns
        else pd.DataFrame()
    )

    suspicious_cross_day_days = pd.DataFrame()
    if not day_minute_summary.empty:
        # Umbral conservador: si hay indicadores como window 60,
        # no deberíamos ver minutos muy tempranos sobrevivir tras dropna total
        suspicious_cross_day_days = day_minute_summary[
            day_minute_summary["first_minute"] < 60
        ].copy()

    # 5.6 Duplicados temporales
    duplicated_timestamps = int(df.index.duplicated().sum())

    # ============================================================
    # 6) Reporte
    # ============================================================
    report = {
        "loaded_from": loaded_from,
        "n_rows_final": int(len(df)),
        "n_cols_final": int(df.shape[1]),
        "n_indicator_columns": int(len(indicator_columns)),
        "is_sorted_chronologically": bool(is_sorted),
        "duplicated_timestamps": duplicated_timestamps,
        "n_indicators_with_nan": int(len(indicators_with_nan)),
        "invalid_regime_rows": invalid_regime_rows,
        "n_suspicious_cross_day_days": int(len(suspicious_cross_day_days)),
    }

    if loaded_from == "computed":
        report["n_rows_before_dropna"] = int(n_before_dropna)
        report["n_rows_after_dropna"] = int(n_after_dropna)
        report["rows_removed_by_dropna"] = int(n_before_dropna - n_after_dropna)

    if print_report:
        print("=" * 100)
        print("DATASET FINAL CON INDICADORES TÉCNICOS")
        print("=" * 100)
        print(f"Origen: {report['loaded_from']}")
        print(f"Filas finales: {report['n_rows_final']}")
        print(f"Columnas finales: {report['n_cols_final']}")
        print(f"Cantidad de indicadores técnicos: {report['n_indicator_columns']}")
        if "n_rows_before_dropna" in report:
            print(f"Filas antes de dropna final: {report['n_rows_before_dropna']}")
            print(f"Filas después de dropna final: {report['n_rows_after_dropna']}")
            print(f"Filas eliminadas por NaNs: {report['rows_removed_by_dropna']}")
        print(f"Orden cronológico correcto: {report['is_sorted_chronologically']}")
        print(f"Timestamps duplicados: {report['duplicated_timestamps']}")
        print(f"Indicadores con NaNs remanentes: {report['n_indicators_with_nan']}")
        print(f"Filas con flags de régimen inválidos: {report['invalid_regime_rows']}")
        print(f"Días sospechosos de cruce entre días: {report['n_suspicious_cross_day_days']}")
        print()

        print("-" * 100)
        print("FILAS POR RÉGIMEN DE MERCADO")
        print("-" * 100)
        print(regime_summary.to_string(index=False))
        print()

        print("-" * 100)
        print("LISTADO TOTAL DE INDICADORES TÉCNICOS")
        print("-" * 100)
        print(indicator_columns)
        print()

        if len(indicators_with_nan) == 0:
            print("[OK] No quedan NaNs en los indicadores técnicos.")
        else:
            print("[WARN] Quedan NaNs en algunos indicadores:")
            print(indicators_with_nan.to_string())
            print()

        if is_sorted:
            print("[OK] El dataset final está en orden cronológico.")
        else:
            print("[WARN] El dataset final NO está en orden cronológico.")

        if duplicated_timestamps == 0:
            print("[OK] No hay timestamps duplicados.")
        else:
            print(f"[WARN] Hay {duplicated_timestamps} timestamps duplicados.")

        if len(suspicious_cross_day_days) == 0:
            print("[OK] No se detectaron señales evidentes de cruce entre días en indicadores.")
        else:
            print("[WARN] Hay días potencialmente sospechosos respecto al arranque de indicadores:")
            print(suspicious_cross_day_days.head(20).to_string(index=False))

    return df, indicator_columns, {
        "report": report,
        "regime_summary": regime_summary,
        "indicator_columns": indicator_columns,
        "nan_counts_indicators": indicators_with_nan,
        "suspicious_cross_day_days": suspicious_cross_day_days,
        "day_minute_summary": day_minute_summary,
    }

In [37]:
mnq_intraday_with_indicators, indicator_columns, indicators_report = load_or_build_mnq_intraday_with_indicators(
    mnq_intraday_targets,
    print_report=True,
)

[OK] Calculando indicadores técnicos...
[OK] Calculado y guardado: /content/drive/MyDrive/neural_profit/data/targets/mnq_intraday_with_indicators.parquet
DATASET FINAL CON INDICADORES TÉCNICOS
Origen: computed
Filas finales: 779590
Columnas finales: 61
Cantidad de indicadores técnicos: 42
Filas antes de dropna final: 894845
Filas después de dropna final: 779590
Filas eliminadas por NaNs: 115255
Orden cronológico correcto: True
Timestamps duplicados: 0
Indicadores con NaNs remanentes: 0
Filas con flags de régimen inválidos: 0
Días sospechosos de cruce entre días: 0

----------------------------------------------------------------------------------------------------
FILAS POR RÉGIMEN DE MERCADO
----------------------------------------------------------------------------------------------------
 regime_flag  n_rows
is_overnight  196840
is_premarket   77700
  is_opening   77700
  is_regular  388500
  is_closing   38850

----------------------------------------------------------------------

Filtramos todos los NaNs del dataset

In [38]:
# Verificación posterior
#start_time_full_day = info_mnq_indicators["datetime_min"][11:16]
#final_time_full_day = info_mnq_indicators["datetime_max"][11:16]

In [39]:
#start_time_full_day, final_time_full_day

# **3. Separación IS vs OOS**

En esta etapa se realiza la separación del dataset en dos subconjuntos temporales: **in-sample (IS)** y **out-of-sample (OOS)**.
Esta división se efectúa de manera estrictamente cronológica, respetando el orden temporal de los datos y evitando cualquier tipo de filtración de información futura.

El conjunto **in-sample (IS)** se utiliza para:

* el cálculo y selección de indicadores técnicos,
* el análisis del Information Coefficient (IC),
* la evaluación de dependencias y colinealidad entre features.

El conjunto **out-of-sample (OOS)** se reserva exclusivamente para:

* validar la estabilidad temporal de las relaciones observadas,
* comprobar que la capacidad predictiva de los indicadores no es producto del sobreajuste,
* verificar que las señales seleccionadas mantienen poder explicativo en datos no vistos.

Esta separación es un paso crítico para garantizar la validez estadística del proceso de feature engineering.
Los indicadores se seleccionan en base a su desempeño en el conjunto IS y se validan posteriormente en OOS.
Un indicador solo se considera robusto si mantiene un comportamiento consistente fuera de muestra, tanto en el signo como en la estabilidad de la magnitud del IC.

De este modo, la selección final de features se basa en criterios de **robustez temporal**, y no únicamente en el desempeño observado dentro del período de entrenamiento.

**Criterio propuesto (ajustable)**

- IS: desde 2019-12-23 hasta 2022-12-31
- OOS: desde 2023-01-01 hasta 2025-06-13

Este split es solo para selección y validación de features, no es el split final de modelado.

In [42]:
import numpy as np
import pandas as pd

def add_is_oos_split(
    df: pd.DataFrame,
    *,
    cut_date: str | pd.Timestamp = "2022-12-31",
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    split_col: str = "split_fe",
    sort_before_split: bool = True,
    overwrite: bool = True,
    print_report: bool = True,
) -> pd.DataFrame:
    """
    Agrega una columna de split IS/OOS al dataset de entrada.

    Parámetros
    ----------
    df : pd.DataFrame
        Dataset de entrada, en este caso mnq_intraday_with_indicators.
    cut_date : str | pd.Timestamp
        Fecha de corte. <= cut_date -> IS, > cut_date -> OOS.
    date_col : str
        Nombre de la columna de fecha.
    minute_col : str
        Nombre de la columna de minuto intradía.
    split_col : str
        Nombre de la columna de salida para el split.
    sort_before_split : bool
        Si True, ordena cronológicamente antes de crear el split.
    overwrite : bool
        Si False y split_col ya existe, lanza error.
    print_report : bool
        Si True, imprime un resumen del split.

    Devuelve
    --------
    pd.DataFrame
        DataFrame con la columna split_fe agregada.
    """
    out = df.copy()

    if split_col in out.columns and not overwrite:
        raise ValueError(f"La columna '{split_col}' ya existe y overwrite=False.")

    if date_col not in out.columns:
        raise KeyError(f"Falta la columna '{date_col}' en el DataFrame.")

    out[date_col] = pd.to_datetime(out[date_col])
    cut_date = pd.to_datetime(cut_date)

    # ============================================================
    # 1) Orden cronológico
    # ============================================================
    if sort_before_split:
        if isinstance(out.index, pd.DatetimeIndex):
            out = out.sort_index().copy()
        else:
            sort_cols = [date_col]
            if minute_col in out.columns:
                sort_cols.append(minute_col)
            out = out.sort_values(by=sort_cols).copy()

    # Validación
    if isinstance(out.index, pd.DatetimeIndex):
        is_sorted = out.index.is_monotonic_increasing
    else:
        if minute_col in out.columns:
            is_sorted = (
                out[[date_col, minute_col]]
                .reset_index(drop=True)
                .equals(
                    out[[date_col, minute_col]]
                    .sort_values([date_col, minute_col])
                    .reset_index(drop=True)
                )
            )
        else:
            is_sorted = out[date_col].is_monotonic_increasing

    if not is_sorted:
        raise ValueError("El dataset no está ordenado cronológicamente antes del split.")

    # ============================================================
    # 2) Crear split IS / OOS
    # ============================================================
    out[split_col] = np.where(
        out[date_col] <= cut_date,
        "IS",
        "OOS",
    )

    # ============================================================
    # 3) Reporte
    # ============================================================
    if print_report:
        split_summary = (
            out.groupby(split_col)
            .agg(
                n_rows=(split_col, "size"),
                start_date=(date_col, "min"),
                end_date=(date_col, "max"),
                n_days=(date_col, "nunique"),
            )
            .reset_index()
        )

        print("=" * 90)
        print("SEPARACIÓN IS vs OOS")
        print("=" * 90)
        print(f"Fecha de corte: {cut_date.date()}")
        print(f"Orden cronológico correcto: {is_sorted}")
        print()
        print(split_summary.to_string(index=False))

    return out


In [43]:
mnq_intraday_with_indicators = add_is_oos_split(
    mnq_intraday_with_indicators,
    cut_date="2022-12-31",
    print_report=True,
)

SEPARACIÓN IS vs OOS
Fecha de corte: 2022-12-31
Orden cronológico correcto: True

split_fe  n_rows start_date   end_date  n_days
      IS  426818 2020-01-02 2022-12-30     709
     OOS  352772 2023-01-03 2025-06-13     586


#**4. Evaluación mediante Information Coefficient (IC)**

## **4.1. Marco teórico**

### **4.1.1. Target `delta_h`**

En el dataset se utilizan como variables objetivo los targets continuos delta_60 y delta_90, que representan la variación futura del precio en puntos entre el instante ( t ) y ( t+h ).

Estos targets capturan simultáneamente dirección y magnitud del movimiento futuro, lo que los hace adecuados para evaluar la capacidad predictiva de indicadores técnicos.

El análisis se realiza mediante el coeficiente de correlación de Spearman, ya que:

* no asume relaciones lineales,
* es robusto a outliers,
* captura relaciones monotónicas, más acordes con datos financieros intradía.

Interpretación:

* IC > 0 → el indicador se asocia positivamente con el movimiento futuro
* IC < 0 → relación inversa


### **4.1.2. Criterio metodológico — Evaluación de IC por régimen de mercado**


El IC se calcula de forma separada por régimen de mercado (premarket, opening, regular, closing, overnight), manteniendo en todos los casos la división IS / OOS.

Este enfoque permite:

* identificar factores robustos que mantienen señal en distintos regímenes,
* detectar dependencias contextuales intradía,
* evaluar la estabilidad de la señal en condiciones heterogéneas del mercado.

De esta forma, se prioriza la robustez de los factores y se evita optimizar prematuramente sobre condiciones específicas.



### **4.1.3. Uso de los resultados**

Las tablas de IC permiten:

* identificar indicadores con señal OOS consistente,
* analizar diferencias entre regímenes,
* evaluar la estabilidad IS vs OOS.



### **4.1.4. Síntesis**


El Information Coefficient permite medir de forma directa la capacidad predictiva de los indicadores técnicos.

Su evaluación por régimen y fuera de muestra constituye un criterio central para la selección de factores robustos y generalizables.

## **4.2. Implementación de cálculo de IC**

### **4.2.1. Marcas temporales de régimen**

In [47]:
import pandas as pd

def build_regime_ranges_table(
    df: pd.DataFrame,
    *,
    minute_col: str = "minute_of_day",
    regime_flags=("is_overnight", "is_premarket", "is_opening", "is_regular", "is_closing"),
) -> pd.DataFrame:
    """
    Construye una tabla resumen con:
    - régimen
    - minuto inicial
    - minuto final
    - hora inicial HH:MM
    - hora final HH:MM
    - cantidad de filas
    """
    if minute_col not in df.columns:
        raise KeyError(f"Falta la columna '{minute_col}' en el DataFrame.")

    rows = []

    for flag in regime_flags:
        if flag not in df.columns:
            raise KeyError(f"Falta la columna '{flag}' en el DataFrame.")

        m = df.loc[df[flag].fillna(0).astype(int) == 1, minute_col].dropna()

        if m.empty:
            rows.append({
                "regime_flag": flag,
                "start_minute_of_day": pd.NA,
                "end_minute_of_day": pd.NA,
                "start_hhmm": pd.NA,
                "end_hhmm": pd.NA,
                "n_rows": 0,
            })
        else:
            start_min = int(m.min())
            end_min = int(m.max())

            rows.append({
                "regime_flag": flag,
                "start_minute_of_day": start_min,
                "end_minute_of_day": end_min,
                "start_hhmm": f"{start_min // 60:02d}:{start_min % 60:02d}",
                "end_hhmm": f"{end_min // 60:02d}:{end_min % 60:02d}",
                "n_rows": int(m.shape[0]),
            })

    out = pd.DataFrame(rows)

    out = out.sort_values(
        by="start_minute_of_day",
        na_position="last"
    ).reset_index(drop=True)

    return out[
        [
            "regime_flag",
            "start_minute_of_day",
            "end_minute_of_day",
            "start_hhmm",
            "end_hhmm",
            "n_rows",
        ]
    ]

In [48]:
regime_ranges = build_regime_ranges_table(mnq_intraday_with_indicators)
regime_ranges

,regime_flag,start_minute_of_day,end_minute_of_day,start_hhmm,end_hhmm,n_rows
0,is_overnight,359,960,05:59,16:00,196840
1,is_premarket,510,569,08:30,09:29,77700
2,is_opening,570,629,09:30,10:29,77700
3,is_regular,630,929,10:30,15:29,388500
4,is_closing,930,959,15:30,15:59,38850


### **4.2.2. Funciones para calcular IC**

In [49]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# =========================
# Utilidades base
# =========================

def spearman_ic(x: pd.Series, y: pd.Series) -> float:
    """IC Spearman entre x e y, ignorando NaNs."""
    mask = x.notna() & y.notna()
    if mask.sum() < 3:
        return np.nan
    return spearmanr(x[mask], y[mask]).correlation


def filter_market_regime(
    df: pd.DataFrame,
    regime_flag: str | None = None,
) -> pd.DataFrame:
    """
    Filtra el DataFrame por régimen de mercado.

    Parámetros
    ----------
    regime_flag : str | None
        - None  -> no filtra, usa toda la jornada
        - str   -> nombre de columna binaria del régimen, por ejemplo:
                   'is_overnight', 'is_premarket', 'is_opening',
                   'is_regular', 'is_closing'
    """
    if regime_flag is None:
        return df.copy()

    if regime_flag not in df.columns:
        raise KeyError(f"Falta la columna de régimen '{regime_flag}' en el DataFrame.")

    return df.loc[df[regime_flag].fillna(0).astype(int) == 1].copy()


def daily_ic(
    df: pd.DataFrame,
    indicator_col: str,
    target_col: str,
    date_col: str = "date",
) -> pd.Series:
    """IC Spearman por día."""
    if date_col not in df.columns:
        raise ValueError(f"Falta la columna '{date_col}' en df.")

    return df.groupby(date_col).apply(
        lambda g: spearman_ic(g[indicator_col], g[target_col])
    )


# =========================
# Tabla IC (indicadores × delta_h) con IS/OOS por régimen
# =========================

def compute_ic_table_is_oos_by_regime(
    df: pd.DataFrame,
    indicator_columns: list[str],
    *,
    horizons: tuple[int, ...] = (60, 90),
    regime_flag: str | None = None,      # None = full day
    use_daily_ic: bool = True,
    split_col: str = "split_fe",         # "IS" / "OOS"
    date_col: str = "date",
) -> pd.DataFrame:
    """
    Calcula IC Spearman entre cada indicador y los targets delta_h,
    separando IS y OOS, para toda la jornada o para un régimen específico.

    Targets esperados:
      - delta_60
      - delta_90

    Devuelve:
      - indicator, horizon, target_col, regime_flag
      - IC_IS, IC_OOS, oos_minus_is
      - n_pairs_IS, n_pairs_OOS
      - abs_IC_OOS
      - note
    """
    # 1) Filtrado por régimen
    dfr = filter_market_regime(df, regime_flag=regime_flag)

    # 2) Validaciones estructurales
    required = {split_col, date_col}
    missing_req = [c for c in required if c not in dfr.columns]
    if missing_req:
        raise ValueError(f"Faltan columnas requeridas: {missing_req}")

    # 3) Subsets IS / OOS
    df_is = dfr[dfr[split_col] == "IS"].copy()
    df_oos = dfr[dfr[split_col] == "OOS"].copy()

    rows = []

    for ind in indicator_columns:
        for h in horizons:
            target_col = f"delta_{h}"

            # Validación de columnas
            missing = [c for c in (ind, target_col) if c not in dfr.columns]
            if missing:
                rows.append({
                    "indicator": ind,
                    "horizon": h,
                    "target_col": target_col,
                    "regime_flag": regime_flag if regime_flag is not None else "full_day",
                    "IC_IS": np.nan,
                    "IC_OOS": np.nan,
                    "oos_minus_is": np.nan,
                    "n_pairs_IS": 0,
                    "n_pairs_OOS": 0,
                    "abs_IC_OOS": np.nan,
                    "note": f"missing: {missing}",
                })
                continue

            # Conteo de pares válidos
            n_pairs_is = int((df_is[ind].notna() & df_is[target_col].notna()).sum())
            n_pairs_oos = int((df_oos[ind].notna() & df_oos[target_col].notna()).sum())

            # --- IS ---
            if use_daily_ic:
                ic_is_series = daily_ic(df_is, ind, target_col, date_col=date_col)
                ic_is = float(ic_is_series.mean()) if len(ic_is_series) > 0 else np.nan
            else:
                ic_is = float(spearman_ic(df_is[ind], df_is[target_col]))

            # --- OOS ---
            if use_daily_ic:
                ic_oos_series = daily_ic(df_oos, ind, target_col, date_col=date_col)
                ic_oos = float(ic_oos_series.mean()) if len(ic_oos_series) > 0 else np.nan
            else:
                ic_oos = float(spearman_ic(df_oos[ind], df_oos[target_col]))

            rows.append({
                "indicator": ind,
                "horizon": h,
                "target_col": target_col,
                "regime_flag": regime_flag if regime_flag is not None else "full_day",
                "IC_IS": ic_is,
                "IC_OOS": ic_oos,
                "oos_minus_is": (ic_oos - ic_is) if (pd.notna(ic_is) and pd.notna(ic_oos)) else np.nan,
                "n_pairs_IS": n_pairs_is,
                "n_pairs_OOS": n_pairs_oos,
                "abs_IC_OOS": abs(ic_oos) if pd.notna(ic_oos) else np.nan,
                "note": "",
            })

    out = pd.DataFrame(rows)

    out = (
        out.sort_values(
            ["regime_flag", "horizon", "abs_IC_OOS"],
            ascending=[True, True, False]
        )
        .reset_index(drop=True)
    )

    return out

### **4.2.3. Función para calculo de IC tables**

In [50]:
import os
import json
import pandas as pd

# ============================================================
# Cache de IC tables (Google Drive)
# ============================================================

CACHE_DIR_DELTA = "/content/drive/MyDrive/neural_profit/data/targets/ic_tables_delta"
os.makedirs(CACHE_DIR_DELTA, exist_ok=True)


def _select_cache_dir() -> str:
    """
    Devuelve el directorio de cache para IC tables de delta.
    """
    return CACHE_DIR_DELTA


def _ic_cache_paths(cache_dir: str, name: str) -> dict:
    """
    Devuelve paths de cache para una ic_table:
      - parquet: datos
      - json: metadatos
    """
    return {
        "data": os.path.join(cache_dir, f"{name}.parquet"),
        "meta": os.path.join(cache_dir, f"{name}.meta.json"),
    }


def load_or_compute_ic_table(
    *,
    name: str,
    df: pd.DataFrame,
    indicator_columns: list[str],
    horizons: tuple[int, ...] = (60, 90),
    regime_flag: str | None = None,   # None = full day
    use_daily_ic: bool = True,
    split_col: str = "split_fe",
    date_col: str = "date",
    force_recompute: bool = False,
) -> pd.DataFrame:
    """
    Carga una ic_table desde cache si existe; si no existe, la calcula y la guarda.

    Parámetros
    ----------
    name : str
        Identificador del artefacto en cache.
    df : pd.DataFrame
        DataFrame con indicadores + targets + columnas auxiliares.
    indicator_columns : list[str]
        Lista de indicadores a evaluar.
    horizons : tuple[int, ...]
        Horizontes a evaluar. Ej.: (60, 90).
    regime_flag : str | None
        Régimen de mercado a evaluar.
        - None -> jornada completa
        - 'is_overnight', 'is_premarket', 'is_opening',
          'is_regular', 'is_closing'
    use_daily_ic : bool
        Si True, calcula IC diario y luego promedia.
        Si False, calcula IC global.
    split_col : str
        Columna de split IS/OOS.
    date_col : str
        Columna de fecha para agrupar daily IC.
    force_recompute : bool
        Si True, ignora cache y recalcula.
    """
    cache_dir = _select_cache_dir()
    paths = _ic_cache_paths(cache_dir, name)

    # 1) Si existe cache y no forzamos recálculo -> cargar
    if (not force_recompute) and os.path.exists(paths["data"]):
        ic_table = pd.read_parquet(paths["data"])
        print(f"[cache] Loaded: {paths['data']}")
        return ic_table

    # 2) Calcular ic_table
    ic_table = compute_ic_table_is_oos_by_regime(
        df=df,
        indicator_columns=indicator_columns,
        horizons=horizons,
        regime_flag=regime_flag,
        use_daily_ic=use_daily_ic,
        split_col=split_col,
        date_col=date_col,
    )

    # 3) Guardar datos
    ic_table.to_parquet(paths["data"], index=False)

    # 4) Guardar metadatos mínimos
    meta = {
        "name": name,
        "target_type": "delta",
        "horizons": list(horizons),
        "regime_flag": regime_flag if regime_flag is not None else "full_day",
        "use_daily_ic": use_daily_ic,
        "split_col": split_col,
        "date_col": date_col,
        "n_indicators": len(indicator_columns),
        "cache_dir": cache_dir,
    }

    with open(paths["meta"], "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    print(f"[cache] Computed & saved: {paths['data']}")
    return ic_table


### **4.2.4. Aplicación de cálculos**

In [51]:
def build_all_ic_tables(
    df: pd.DataFrame,
    *,
    indicator_columns: list[str],
    horizons: tuple[int, ...] = (60, 90),
    use_daily_ic: bool = True,
    split_col: str = "split_fe",
    date_col: str = "date",
    force_recompute: bool = False,
) -> dict[str, pd.DataFrame]:
    """
    Calcula o carga todas las tablas IC por régimen.

    Devuelve un diccionario con:
    - full_day
    - is_overnight
    - is_premarket
    - is_opening
    - is_regular
    - is_closing
    """
    regime_map = {
        "full_day": None,
        "is_overnight": "is_overnight",
        "is_premarket": "is_premarket",
        "is_opening": "is_opening",
        "is_regular": "is_regular",
        "is_closing": "is_closing",
    }

    ic_tables = {}

    for name, regime_flag in regime_map.items():
        ic_tables[name] = load_or_compute_ic_table(
            name=f"ic_table_{name}",
            df=df,
            indicator_columns=indicator_columns,
            horizons=horizons,
            regime_flag=regime_flag,
            use_daily_ic=use_daily_ic,
            split_col=split_col,
            date_col=date_col,
            force_recompute=force_recompute,
        )

    return ic_tables

In [52]:
ic_tables = build_all_ic_tables(
    mnq_intraday_with_indicators,
    indicator_columns=indicator_columns,
    horizons=(60, 90),
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
    force_recompute=False,
)

[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_delta/ic_table_full_day.parquet
[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_delta/ic_table_is_overnight.parquet
[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_delta/ic_table_is_premarket.parquet
[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_delta/ic_table_is_opening.parquet
[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_delta/ic_table_is_regular.parquet
[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_delta/ic_table_is_closing.parquet


In [53]:
ic_table_full_day   = ic_tables["full_day"]
ic_table_overnight  = ic_tables["is_overnight"]
ic_table_premarket  = ic_tables["is_premarket"]
ic_table_opening    = ic_tables["is_opening"]
ic_table_regular    = ic_tables["is_regular"]
ic_table_closing    = ic_tables["is_closing"]

## **4.3. Análisis de resultados**

Qué vamos a analizar primero:

- Magnitud del IC_OOS → ¿hay señal?
- Signo del IC_OOS → ¿momentum o reversión?
- Top indicadores por régimen → ¿se repiten o cambian?

### **Código**


In [54]:
import pandas as pd

def quick_ic_observation(
    ic_tables: dict[str, pd.DataFrame],
    *,
    horizon: int,
    top_n: int = 5,
) -> None:
    """
    Análisis simple de IC:
    - Top indicadores por |IC_OOS|
    - Signo predominante
    - Magnitud promedio de IC_OOS
    """

    print("\n" + "=" * 90)
    print(f"ANÁLISIS SIMPLE IC | delta_{horizon}")
    print("=" * 90)

    for regime_name, df in ic_tables.items():
        x = df[df["horizon"] == horizon].copy()

        x = x[x["IC_OOS"].notna()]

        if x.empty:
            continue

        x["abs_IC_OOS"] = x["IC_OOS"].abs()

        top = x.sort_values("abs_IC_OOS", ascending=False).head(top_n)

        mean_ic = x["IC_OOS"].mean()
        mean_abs_ic = x["abs_IC_OOS"].mean()

        pct_positive = (x["IC_OOS"] > 0).mean() * 100
        pct_negative = (x["IC_OOS"] < 0).mean() * 100

        print("\n" + "-" * 90)
        print(f"Regimen: {regime_name}")
        print("-" * 90)

        print(f"IC_OOS medio: {mean_ic:.4f}")
        print(f"|IC_OOS| medio: {mean_abs_ic:.4f}")
        print(f"% IC positivo: {pct_positive:.1f}%")
        print(f"% IC negativo: {pct_negative:.1f}%")

        print("\nTop indicadores:")
        print(
            top[[
                "indicator",
                "IC_OOS",
                "abs_IC_OOS"
            ]].to_string(index=False)
        )

In [55]:
quick_ic_observation(ic_tables, horizon=60)
quick_ic_observation(ic_tables, horizon=90)


ANÁLISIS SIMPLE IC | delta_60

------------------------------------------------------------------------------------------
Regimen: full_day
------------------------------------------------------------------------------------------
IC_OOS medio: -0.0488
|IC_OOS| medio: 0.0684
% IC positivo: 23.8%
% IC negativo: 76.2%

Top indicadores:
indicator    IC_OOS  abs_IC_OOS
   ema_60 -0.130567    0.130567
   roc_60 -0.123252    0.123252
 bb_60_15 -0.109849    0.109849
 bb_60_20 -0.109849    0.109849
 bb_60_25 -0.109849    0.109849

------------------------------------------------------------------------------------------
Regimen: is_overnight
------------------------------------------------------------------------------------------
IC_OOS medio: -0.1617
|IC_OOS| medio: 0.1831
% IC positivo: 23.8%
% IC negativo: 76.2%

Top indicadores:
indicator    IC_OOS  abs_IC_OOS
   ema_60 -0.370539    0.370539
   roc_60 -0.359776    0.359776
 bb_60_15 -0.310131    0.310131
 bb_60_20 -0.310131    0.310131
 

### **Observaciones**


**1. Observaciones generales (delta_60 y delta_90)**

* Existe señal predictiva en los indicadores técnicos.
  Los valores de |IC_OOS| son consistentemente mayores a 0.05 y en muchos casos superiores a 0.1, lo que indica señal débil pero explotable.

* La señal es predominantemente negativa.
  En todos los regímenes, entre el 75% y 88% de los indicadores presentan IC negativo, lo que sugiere un comportamiento de **mean reversion**.


**2. Diferencias por régimen de mercado**

* El régimen de **opening** presenta la señal más fuerte.
  Los valores de IC_OOS alcanzan magnitudes muy altas (hasta ~0.7), lo que indica una fuerte capacidad predictiva en este tramo.

* Los regímenes **overnight y premarket** también muestran señal clara.
  Con IC medios entre -0.15 y -0.18, evidencian que existe estructura predictiva antes de la apertura.

* El régimen **regular** presenta señal moderada.
  La magnitud del IC disminuye, lo que sugiere un mercado más eficiente durante este periodo.

* En la **jornada completa**, la señal se diluye.
  Esto es esperable, ya que mezcla distintos regímenes con comportamientos heterogéneos.


**3. Consistencia de indicadores**

* Los mismos indicadores dominan en todos los regímenes:
  **EMA, ROC y Bollinger Bands** aparecen sistemáticamente como los más relevantes.

* Esto sugiere la existencia de **factores estructurales**, no dependientes de un único régimen.


**4. Comparación delta_60 vs delta_90**

* Los resultados son altamente consistentes entre ambos horizontes.
  Los mismos indicadores aparecen en el top y con magnitudes similares.

* En general, **delta_90 presenta ligeramente mayor magnitud de señal**, lo que indica que horizontes más largos capturan mejor la estructura del mercado.

**5. Conclusión preliminar**

* Hay evidencia clara de señal predictiva en los indicadores técnicos.
* La señal es mayormente de tipo **mean reversion**.
* La intensidad de la señal depende fuertemente del régimen de mercado, siendo más fuerte en apertura.
* Existen indicadores robustos que se mantienen relevantes en múltiples contextos.



# **5. Análisis de robustez IS vs OOS**


El objetivo es medir qué tan bien se mantiene la señal fuera de muestra. Esto es clave porque, en ML para trading, un modelo o factor puede verse bien en entrenamiento y degradarse al generalizar; eso es precisamente lo que se quiere detectar al diagnosticar overfitting y validar desempeño out-of-sample.

La lógica del análisis sería:

- gap pequeño entre IC_IS e IC_OOS → señal más estable
- gap grande → posible inestabilidad o sobreajuste
- mismo signo en IS y OOS → la relación se conserva
- cambio de signo → señal sospechosa, aunque el abs_IC_OOS sea alto

## **5.1. Implementación de código**

In [57]:
import numpy as np
import pandas as pd

def analyze_ic_robustness(
    ic_table: pd.DataFrame,
    *,
    regime_name: str,
    min_abs_ic_oos: float = 0.05,
    require_same_sign: bool = True,
    top_n: int = 10,
) -> dict[str, pd.DataFrame]:
    """
    Analiza robustez IS vs OOS para una tabla IC de un régimen.

    Devuelve:
    ---------
    {
        "summary_df": resumen por horizonte,
        "top_df": top indicadores robustos por horizonte
    }
    """

    required = [
        "indicator", "horizon", "target_col",
        "IC_IS", "IC_OOS"
    ]
    missing = [c for c in required if c not in ic_table.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}")

    df = ic_table.copy()
    df = df[df["IC_IS"].notna() & df["IC_OOS"].notna()].copy()

    if df.empty:
        empty_summary = pd.DataFrame([{
            "regime_name": regime_name,
            "horizon": pd.NA,
            "n_total": 0,
            "n_after_filter": 0,
            "pct_retained": np.nan,
            "mean_IC_IS": np.nan,
            "mean_IC_OOS": np.nan,
            "mean_abs_IC_OOS": np.nan,
            "mean_abs_gap": np.nan,
            "pct_same_sign": np.nan,
            "best_indicator": pd.NA,
            "best_IC_OOS": np.nan,
            "best_abs_gap": np.nan,
            "best_robustness_score": np.nan,
        }])
        empty_top = pd.DataFrame(columns=[
            "regime_name", "indicator", "horizon", "target_col",
            "IC_IS", "IC_OOS", "oos_minus_is", "abs_gap",
            "same_sign", "abs_IC_OOS", "strength_ratio",
            "robustness_score"
        ])
        return {"summary_df": empty_summary, "top_df": empty_top}

    # Variables base
    df["oos_minus_is"] = df["IC_OOS"] - df["IC_IS"]
    df["abs_gap"] = df["oos_minus_is"].abs()
    df["same_sign"] = np.sign(df["IC_IS"]) == np.sign(df["IC_OOS"])
    df["abs_IC_IS"] = df["IC_IS"].abs()
    df["abs_IC_OOS"] = df["IC_OOS"].abs()

    df["strength_ratio"] = np.where(
        df["abs_IC_IS"] > 1e-12,
        df["abs_IC_OOS"] / df["abs_IC_IS"],
        np.nan,
    )

    sign_bonus = np.where(df["same_sign"], 1.0, 0.0)

    # Score robustez
    df["robustness_score"] = (
        0.70 * df["abs_IC_OOS"]
        - 0.25 * df["abs_gap"]
        + 0.05 * sign_bonus
    )

    # Conteo previo por horizonte
    n_total_by_h = (
        df.groupby("horizon")
          .size()
          .rename("n_total")
          .reset_index()
    )

    # Filtro mínimo de señal
    df = df[df["abs_IC_OOS"] >= min_abs_ic_oos].copy()

    # Filtro opcional por consistencia de signo
    if require_same_sign:
        df = df[df["same_sign"]].copy()

    # Si tras filtrar queda vacío
    if df.empty:
        summary_rows = []
        for _, row in n_total_by_h.iterrows():
            summary_rows.append({
                "regime_name": regime_name,
                "horizon": row["horizon"],
                "n_total": int(row["n_total"]),
                "n_after_filter": 0,
                "pct_retained": 0.0,
                "mean_IC_IS": np.nan,
                "mean_IC_OOS": np.nan,
                "mean_abs_IC_OOS": np.nan,
                "mean_abs_gap": np.nan,
                "pct_same_sign": np.nan,
                "best_indicator": pd.NA,
                "best_IC_OOS": np.nan,
                "best_abs_gap": np.nan,
                "best_robustness_score": np.nan,
            })
        return {
            "summary_df": pd.DataFrame(summary_rows),
            "top_df": pd.DataFrame(columns=[
                "regime_name", "indicator", "horizon", "target_col",
                "IC_IS", "IC_OOS", "oos_minus_is", "abs_gap",
                "same_sign", "abs_IC_OOS", "strength_ratio",
                "robustness_score"
            ])
        }

    # Orden final
    df = df.sort_values(
        ["horizon", "robustness_score", "abs_IC_OOS", "abs_gap"],
        ascending=[True, False, False, True]
    ).reset_index(drop=True)

    df["regime_name"] = regime_name

    # Top robustos por horizonte
    top_df = (
        df.groupby("horizon", group_keys=False)
          .head(top_n)
          .reset_index(drop=True)
    )[
        [
            "regime_name", "indicator", "horizon", "target_col",
            "IC_IS", "IC_OOS", "oos_minus_is", "abs_gap",
            "same_sign", "abs_IC_OOS", "strength_ratio",
            "robustness_score"
        ]
    ]

    # Resumen exacto para conclusiones
    summary_rows = []
    for h, g in df.groupby("horizon"):
        n_total = int(n_total_by_h.loc[n_total_by_h["horizon"] == h, "n_total"].iloc[0])
        best = g.iloc[0]

        summary_rows.append({
            "regime_name": regime_name,
            "horizon": h,
            "n_total": n_total,
            "n_after_filter": int(len(g)),
            "pct_retained": float(len(g) / n_total * 100) if n_total > 0 else np.nan,
            "mean_IC_IS": float(g["IC_IS"].mean()),
            "mean_IC_OOS": float(g["IC_OOS"].mean()),
            "mean_abs_IC_OOS": float(g["abs_IC_OOS"].mean()),
            "mean_abs_gap": float(g["abs_gap"].mean()),
            "pct_same_sign": float(g["same_sign"].mean() * 100),
            "best_indicator": best["indicator"],
            "best_IC_OOS": float(best["IC_OOS"]),
            "best_abs_gap": float(best["abs_gap"]),
            "best_robustness_score": float(best["robustness_score"]),
        })

    summary_df = pd.DataFrame(summary_rows).sort_values("horizon").reset_index(drop=True)

    return {
        "summary_df": summary_df,
        "top_df": top_df,
    }

In [58]:
rob_premarket = analyze_ic_robustness(ic_table_premarket, regime_name="premarket")
rob_opening   = analyze_ic_robustness(ic_table_opening,   regime_name="opening")
rob_overnight = analyze_ic_robustness(ic_table_overnight, regime_name="overnight")
rob_regular   = analyze_ic_robustness(ic_table_regular,   regime_name="regular")
rob_closing   = analyze_ic_robustness(ic_table_closing,   regime_name="closing")

## **5.2. Summary all**

In [62]:
robustness_summary_all = pd.concat([
    rob_overnight["summary_df"],
    rob_premarket["summary_df"],
    rob_opening["summary_df"],
    rob_regular["summary_df"],
    rob_closing["summary_df"],
], ignore_index=True)

robustness_summary_all


,regime_name,horizon,n_total,n_after_filter,pct_retained,mean_IC_IS,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign,best_indicator,best_IC_OOS,best_abs_gap,best_robustness_score
0,overnight,60,42,37,88.095238,-0.184732,-0.183992,0.207309,0.008560,100.0,ema_60,-0.370539,0.024267,0.303310
1,overnight,90,42,37,88.095238,-0.185150,-0.174305,0.194988,0.014984,100.0,ema_60,-0.338048,0.036715,0.277455
2,premarket,60,42,32,76.190476,-0.179679,-0.202640,0.202640,0.024066,100.0,ema_60,-0.293864,0.046864,0.243989
3,premarket,90,42,32,76.190476,-0.208407,-0.194013,0.194013,0.015870,100.0,roc_60,-0.284789,0.020058,0.244338
4,opening,60,42,37,88.095238,-0.328104,-0.338748,0.379159,0.017309,100.0,roc_60,-0.671395,0.007656,0.518063
5,opening,90,42,37,88.095238,-0.351982,-0.361387,0.402146,0.014514,100.0,roc_60,-0.706264,0.016245,0.540324
6,regular,60,42,36,85.714286,-0.113207,-0.109746,0.148698,0.011717,100.0,ema_60,-0.253300,0.013310,0.223983
7,regular,90,42,37,88.095238,-0.153170,-0.150431,0.191924,0.008794,100.0,ema_60,-0.326882,0.011298,0.275993
8,closing,<NA>,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Robustez general del sistema**

* La gran mayoría de indicadores sobrevive el filtro de robustez.
  Entre **76% y 88%** de los indicadores se mantienen, lo que indica que la señal **no es frágil**.

* El **100% mantiene el signo entre IS y OOS** en todos los regímenes.
  Esto es muy importante: **no hay reversión de señal**, lo que implica estabilidad estructural.

---

**Gap IS vs OOS**

* El `mean_abs_gap` es **muy bajo en todos los casos** (~0.008 a 0.024).
  Esto indica que:

  * No hay sobreajuste relevante
  * La señal generaliza bien fuera de muestra

---

**Intensidad de la señal por régimen**

* **Opening es el régimen más fuerte**

  * `mean_abs_IC_OOS` ≈ 0.38 – 0.40
  * Mejores scores de robustez (~0.52 – 0.54)
    → Señal muy clara y consistente

* **Overnight y premarket también son fuertes**

  * `mean_abs_IC_OOS` ≈ 0.19 – 0.21
    → Buena señal antes de apertura

* **Regular es el más débil**

  * `mean_abs_IC_OOS` ≈ 0.15 – 0.19
    → Mercado más eficiente, menor predictibilidad

---

**Consistencia entre horizontes**

* Resultados muy similares entre **delta_60 y delta_90**
* En general, **delta_90 muestra levemente mejor estabilidad** (menor gap y buena magnitud)

---

**Mejores indicadores (estructurales)**

* Dominan claramente:

  * `ema_60`
  * `roc_60`

* Se repiten en múltiples regímenes →
  **son factores estructurales, no específicos de un régimen**

---

**Conclusión técnica**

* La señal encontrada es:

  * **real (OOS consistente)**
  * **estable (gap bajo)**
  * **robusta (mismo signo en todos los casos)**

* No hay evidencia de overfitting a nivel de indicadores.

* La predictibilidad está fuertemente concentrada en:

  * **opening (principal)**
  * **overnight / premarket (secundarios)**

* El régimen **regular aporta menos valor predictivo**.


## **5.3. Top indicadores robustos**

In [64]:
robustness_top_all = pd.concat([
    rob_overnight["top_df"],
    rob_premarket["top_df"],
    rob_opening["top_df"],
    rob_regular["top_df"],
    rob_closing["top_df"],
], ignore_index=True)

#robustness_top_all

def find_common_indicators(robustness_top_all: pd.DataFrame) -> pd.DataFrame:
    df = robustness_top_all.copy()

    # Conteo de aparición por indicador
    summary = (
        df.groupby("indicator")
        .agg(
            n_regimes=("regime_name", "nunique"),
            regimes=("regime_name", lambda x: sorted(set(x))),
            mean_IC_OOS=("IC_OOS", "mean"),
            mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
            mean_score=("robustness_score", "mean"),
        )
        .reset_index()
        .sort_values(["n_regimes", "mean_abs_IC_OOS"], ascending=[False, False])
    )

    return summary

common_indicators = find_common_indicators(robustness_top_all)
common_indicators

,indicator,n_regimes,regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_score
8,ema_60,4,"[opening, overnight, premarket, regular]",-0.397697,0.397697,0.322691
11,roc_60,4,"[opening, overnight, premarket, regular]",-0.392403,0.392403,0.319066
10,roc_30,4,"[opening, overnight, premarket, regular]",-0.335522,0.335522,0.278020
7,ema_30,4,"[opening, overnight, premarket, regular]",-0.328344,0.328344,0.274946
3,bb_60_15,4,"[opening, overnight, premarket, regular]",-0.326115,0.326115,0.272138
12,rsi_14,4,"[opening, overnight, premarket, regular]",-0.312140,0.312140,0.264355
6,ema_20,3,"[opening, overnight, premarket]",-0.321288,0.321288,0.270363
14,stoch_k_30,3,"[opening, premarket, regular]",-0.318178,0.318178,0.269408
4,bb_60_20,3,"[overnight, premarket, regular]",-0.269376,0.269376,0.233412
5,bb_60_25,3,"[overnight, premarket, regular]",-0.269376,0.269376,0.233412


**Observaciones**

* Existe un grupo de indicadores claramente **estructurales**:

  * `ema_60`, `roc_60`, `roc_30`, `ema_30`, `bb_60_15`, `rsi_14`
  * Aparecen en **4 regímenes distintos**
    → No dependen del contexto de mercado

* Todos los indicadores presentan **IC negativo consistente**
  → Refuerza la hipótesis de **mean reversion**

* Los indicadores más fuertes en magnitud:

  * `ema_60` y `roc_60` (~0.39)
    → Son los más relevantes globalmente

* Los indicadores con `n_regimes = 3`:

  * Siguen siendo fuertes, pero con cierta dependencia de régimen
  * Ej: `ema_20`, `stoch_k_30`, `bb_60_20`

* Los indicadores con `n_regimes ≤ 2`:

  * Son más **contextuales**
  * Ej: `stoch_k_20`, `roc_20`, bandas cortas (`bb_30_*`)

---

**Lectura estructural**

* Indicadores de **tendencia y momentum suavizado** (EMA, ROC) dominan el ranking
* Indicadores más “ruidosos” o de menor ventana aparecen menos o son específicos

---

**Conclusiones parciales**

* Existen **factores robustos y generalizables**:

  * Principalmente `ema_60` y `roc_60`
  * Funcionan en todos los regímenes relevantes

* La señal no es puntual ni circunstancial
  → Es **consistente en distintos contextos de mercado**

* Se puede separar claramente:

  * **Core features** → `n_regimes ≥ 4`
  * **Features contextuales** → `n_regimes ≤ 2`

* La estructura del problema está dominada por:

  * **momentum / reversión a la media en múltiples escalas**

---

**Conclusión operativa (importante)**

* Ya tienes un **núcleo sólido de features candidatos**:

  * `ema_60`, `roc_60`, `roc_30`, `ema_30`, `bb_60_15`, `rsi_14`

* El resto puede usarse como:

  * features secundarias
  * o específicas por régimen

# **6. Consistencia de signo**

## **6.1. Función General**

In [65]:
import numpy as np
import pandas as pd

def analyze_sign_consistency_across_regimes(
    ic_tables: dict[str, pd.DataFrame],
    *,
    top_n_inconsistent: int = 10,
) -> dict[str, pd.DataFrame]:
    """
    Analiza la consistencia de signo entre IC_IS e IC_OOS
    para múltiples tablas IC por régimen.

    Parámetros
    ----------
    ic_tables : dict[str, pd.DataFrame]
        Diccionario tipo:
        {
            "overnight": ic_table_overnight,
            "premarket": ic_table_premarket,
            ...
        }

    Devuelve
    --------
    {
        "summary_df": resumen por régimen y horizonte,
        "inconsistent_df": indicadores que cambian de signo
    }
    """

    all_rows = []

    for regime_name, df in ic_tables.items():
        required = ["indicator", "horizon", "target_col", "IC_IS", "IC_OOS"]
        missing = [c for c in required if c not in df.columns]
        if missing:
            raise ValueError(f"En régimen '{regime_name}' faltan columnas: {missing}")

        x = df.copy()
        x = x[x["IC_IS"].notna() & x["IC_OOS"].notna()].copy()

        if x.empty:
            continue

        x["regime_name"] = regime_name
        x["same_sign"] = np.sign(x["IC_IS"]) == np.sign(x["IC_OOS"])
        x["oos_minus_is"] = x["IC_OOS"] - x["IC_IS"]
        x["abs_gap"] = x["oos_minus_is"].abs()
        x["abs_IC_OOS"] = x["IC_OOS"].abs()

        all_rows.append(x)

    if not all_rows:
        return {
            "summary_df": pd.DataFrame(),
            "inconsistent_df": pd.DataFrame(),
        }

    all_df = pd.concat(all_rows, ignore_index=True)

    # ============================================================
    # 1) Resumen exacto para conclusiones
    # ============================================================
    summary_df = (
        all_df.groupby(["regime_name", "horizon"], as_index=False)
        .agg(
            n_total=("indicator", "count"),
            n_same_sign=("same_sign", "sum"),
            pct_same_sign=("same_sign", lambda s: float(s.mean() * 100)),
            mean_IC_IS=("IC_IS", "mean"),
            mean_IC_OOS=("IC_OOS", "mean"),
            mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
            mean_abs_gap=("abs_gap", "mean"),
        )
        .sort_values(["horizon", "regime_name"])
        .reset_index(drop=True)
    )

    # ============================================================
    # 2) Indicadores inconsistentes
    # ============================================================
    inconsistent_df = (
        all_df[~all_df["same_sign"]]
        .sort_values(["horizon", "abs_IC_OOS", "abs_gap"], ascending=[True, False, False])
        .reset_index(drop=True)
    )

    if not inconsistent_df.empty:
        inconsistent_df = (
            inconsistent_df.groupby(["regime_name", "horizon"], group_keys=False)
            .head(top_n_inconsistent)
            .reset_index(drop=True)
        )[
            [
                "regime_name",
                "indicator",
                "horizon",
                "target_col",
                "IC_IS",
                "IC_OOS",
                "oos_minus_is",
                "abs_gap",
                "abs_IC_OOS",
                "same_sign",
            ]
        ]

    return {
        "summary_df": summary_df,
        "inconsistent_df": inconsistent_df,
    }

## **6.2. Aplicarlo a todos los regímenes**

In [66]:
ic_tables = {
    "overnight": ic_table_overnight,
    "premarket": ic_table_premarket,
    "opening": ic_table_opening,
    "regular": ic_table_regular,
    "closing": ic_table_closing,
}

sign_results = analyze_sign_consistency_across_regimes(ic_tables)

sign_summary_df = sign_results["summary_df"]
sign_inconsistent_df = sign_results["inconsistent_df"]

sign_summary_df

,regime_name,horizon,n_total,n_same_sign,pct_same_sign,mean_IC_IS,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
0,opening,60,42,42,100.000000,-0.282171,-0.294690,0.337752,0.018390
1,overnight,60,42,41,97.619048,-0.162407,-0.161663,0.183054,0.007719
2,premarket,60,42,39,92.857143,-0.134749,-0.152641,0.157112,0.020831
3,regular,60,42,42,100.000000,-0.095961,-0.093509,0.130009,0.010558
4,opening,90,42,42,100.000000,-0.304070,-0.314481,0.358155,0.014912
5,overnight,90,42,42,100.000000,-0.161800,-0.152631,0.172698,0.013708
6,premarket,90,42,42,100.000000,-0.151939,-0.146559,0.151137,0.018114
7,regular,90,42,42,100.000000,-0.131529,-0.130525,0.171073,0.009157


## **6.3. Observaciones**

**Observaciones**

* La consistencia de signo es **extremadamente alta** en todos los casos:

  * Entre **92.8% y 100%** para todos los regímenes y horizontes

* En **delta_90**, la consistencia es perfecta:

  * **100% en todos los regímenes**
    → Ningún indicador cambia de signo entre IS y OOS

* En **delta_60**, solo hay pequeñas excepciones:

  * Overnight: 97.6% (1 indicador inconsistente)
  * Premarket: 92.8% (3 indicadores inconsistentes)

* El **gap IS vs OOS sigue siendo bajo**:

  * Entre ~0.007 y 0.021
    → Refuerza estabilidad

* La señal mantiene **misma dirección negativa**:

  * Coherente con lo observado antes (mean reversion)

---

**Lectura estructural**

* La dirección de la señal es **estable en el tiempo**
* No hay evidencia de:

  * inversión de señal
  * comportamiento errático

Esto es exactamente lo que se espera de un sistema bien comportado.

---

**Conclusiones parciales**

* La señal es **direccionalmente robusta**:

  * Lo que funciona en IS sigue funcionando en OOS

* **No hay sobreajuste en la dirección de la señal**
  → Punto crítico validado

* **delta_90 es aún más estable que delta_60**
  → Horizonte más confiable desde el punto de vista estructural

* Los pocos indicadores inconsistentes:

  * son marginales
  * no afectan la estructura global

---

**Conclusión técnica**

* La consistencia de signo confirma que:

  * la señal detectada es **real**
  * no es un artefacto del split IS/OOS

* Esto valida completamente el uso de:

  * **IC como métrica**
  * **indicadores técnicos como features**

---

Si lo vemos en conjunto con el punto 5 (robustez), ya tenemos una conclusión fuerte:

→ **hay señal, es estable, y generaliza correctamente OOS**


# **7. Análisis por régimen de mercado**


   
   Se evalúan los indicadores en:

* overnight
* premarket
* opening
* regular
* closing

Objetivo:

* detectar factores estructurales
* detectar factores contextuales

## **7.1. Código general**

In [72]:
import numpy as np
import pandas as pd

def analyze_regime_dependence(
    ic_tables: dict[str, pd.DataFrame],
    *,
    horizons: tuple[int, ...] = (60, 90),
    top_n_each_regime: int = 10,
    min_abs_ic_oos: float = 0.05,
    require_same_sign: bool = True,
) -> dict[str, pd.DataFrame]:
    """
    Analiza factores estructurales y contextuales por régimen de mercado.

    Parámetros
    ----------
    ic_tables : dict[str, pd.DataFrame]
        Ejemplo:
        {
            "overnight": ic_table_overnight,
            "premarket": ic_table_premarket,
            "opening": ic_table_opening,
            "regular": ic_table_regular,
            "closing": ic_table_closing,
        }

    horizons : tuple[int, ...]
        Horizontes a evaluar.
    top_n_each_regime : int
        Cantidad de indicadores top por régimen para comparar entre regímenes.
    min_abs_ic_oos : float
        Filtro mínimo de señal OOS.
    require_same_sign : bool
        Si True, exige consistencia de signo IS/OOS.

    Devuelve
    --------
    {
        "summary_by_regime_df": resumen por régimen y horizonte,
        "structural_factors_df": factores estructurales,
        "contextual_factors_df": factores contextuales,
        "top_by_regime_df": top indicadores por régimen
    }
    """

    required_cols = [
        "indicator", "horizon", "target_col",
        "IC_IS", "IC_OOS", "abs_IC_OOS"
    ]

    prepared_rows = []
    top_rows = []

    for regime_name, df in ic_tables.items():
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"En régimen '{regime_name}' faltan columnas: {missing}")

        x = df.copy()
        x = x[x["IC_IS"].notna() & x["IC_OOS"].notna()].copy()

        if x.empty:
            continue

        x["regime_name"] = regime_name
        x["same_sign"] = np.sign(x["IC_IS"]) == np.sign(x["IC_OOS"])
        x["oos_minus_is"] = x["IC_OOS"] - x["IC_IS"]
        x["abs_gap"] = x["oos_minus_is"].abs()

        # filtro mínimo
        x = x[x["abs_IC_OOS"] >= min_abs_ic_oos].copy()

        if require_same_sign:
            x = x[x["same_sign"]].copy()

        if x.empty:
            continue

        prepared_rows.append(x)

        # top por régimen y horizonte
        for h in horizons:
            xh = x[x["horizon"] == h].copy()
            if xh.empty:
                continue

            top_h = (
                xh.sort_values(["abs_IC_OOS", "abs_gap"], ascending=[False, True])
                  .head(top_n_each_regime)
                  .copy()
            )
            top_rows.append(top_h)

    if not prepared_rows:
        return {
            "summary_by_regime_df": pd.DataFrame(),
            "structural_factors_df": pd.DataFrame(),
            "contextual_factors_df": pd.DataFrame(),
            "top_by_regime_df": pd.DataFrame(),
        }

    prepared_df = pd.concat(prepared_rows, ignore_index=True)

    if top_rows:
        top_by_regime_df = pd.concat(top_rows, ignore_index=True)
    else:
        top_by_regime_df = pd.DataFrame()

    # ============================================================
    # 1) Resumen por régimen
    # ============================================================
    summary_by_regime_df = (
        prepared_df.groupby(["regime_name", "horizon"], as_index=False)
        .agg(
            n_indicators=("indicator", "count"),
            mean_IC_IS=("IC_IS", "mean"),
            mean_IC_OOS=("IC_OOS", "mean"),
            mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
            mean_abs_gap=("abs_gap", "mean"),
            pct_same_sign=("same_sign", lambda s: float(s.mean() * 100)),
            best_indicator=("indicator", "first"),
        )
        .sort_values(["horizon", "mean_abs_IC_OOS"], ascending=[True, False])
        .reset_index(drop=True)
    )

    # ============================================================
    # 2) Factores estructurales
    #    = aparecen en varios regímenes dentro del top
    # ============================================================
    if not top_by_regime_df.empty:
        structural_factors_df = (
            top_by_regime_df.groupby(["horizon", "indicator"], as_index=False)
            .agg(
                n_regimes=("regime_name", "nunique"),
                regimes=("regime_name", lambda s: sorted(set(s))),
                mean_IC_OOS=("IC_OOS", "mean"),
                mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
                mean_abs_gap=("abs_gap", "mean"),
            )
            .sort_values(["horizon", "n_regimes", "mean_abs_IC_OOS"], ascending=[True, False, False])
            .reset_index(drop=True)
        )
    else:
        structural_factors_df = pd.DataFrame()

    # ============================================================
    # 3) Factores contextuales
    #    = aparecen solo en un régimen dentro del top
    # ============================================================
    if not structural_factors_df.empty:
        contextual_only = structural_factors_df[structural_factors_df["n_regimes"] == 1].copy()

        contextual_factors_df = contextual_only.rename(columns={"regimes": "regime"})[
            ["horizon", "indicator", "regime", "mean_IC_OOS", "mean_abs_IC_OOS", "mean_abs_gap"]
        ].sort_values(["horizon", "mean_abs_IC_OOS"], ascending=[True, False]).reset_index(drop=True)
    else:
        contextual_factors_df = pd.DataFrame()

    return {
        "summary_by_regime_df": summary_by_regime_df,
        "structural_factors_df": structural_factors_df,
        "contextual_factors_df": contextual_factors_df,
        "top_by_regime_df": top_by_regime_df,
    }

In [73]:
ic_tables = {
    "overnight": ic_table_overnight,
    "premarket": ic_table_premarket,
    "opening": ic_table_opening,
    "regular": ic_table_regular,
    "closing": ic_table_closing,
}

regime_analysis = analyze_regime_dependence(
    ic_tables,
    horizons=(60, 90),
    top_n_each_regime=10,
    min_abs_ic_oos=0.05,
    require_same_sign=True,
)

## **7.2. Resultados**

In [76]:
print('\n1. Resumen por régimen:\n')
display(regime_analysis["summary_by_regime_df"])
print('\n2. Factores estructurales:\n')
display(regime_analysis["structural_factors_df"])
print('\n3. Factores contextuales:\n')
display(regime_analysis["contextual_factors_df"])
print('\n4. Top usados para el análisis:\n')
display(regime_analysis["top_by_regime_df"])


1. Resumen por régimen:



,regime_name,horizon,n_indicators,mean_IC_IS,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign,best_indicator
0,opening,60,37,-0.328104,-0.338748,0.379159,0.017309,100.0,roc_60
1,overnight,60,37,-0.184732,-0.183992,0.207309,0.008560,100.0,ema_60
2,premarket,60,32,-0.179679,-0.202640,0.202640,0.024066,100.0,ema_60
3,regular,60,36,-0.113207,-0.109746,0.148698,0.011717,100.0,ema_60
4,opening,90,37,-0.351982,-0.361387,0.402146,0.014514,100.0,roc_60
5,overnight,90,37,-0.185150,-0.174305,0.194988,0.014984,100.0,ema_60
6,premarket,90,32,-0.208407,-0.194013,0.194013,0.015870,100.0,roc_60
7,regular,90,37,-0.153170,-0.150431,0.191924,0.008794,100.0,ema_60



2. Factores estructurales:



,horizon,indicator,n_regimes,regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
0,60,ema_60,4,"[opening, overnight, premarket, regular]",-0.389948,0.389948,0.025392
1,60,roc_60,4,"[opening, overnight, premarket, regular]",-0.386956,0.386956,0.024161
2,60,ema_30,4,"[opening, overnight, premarket, regular]",-0.320126,0.320126,0.019899
3,60,roc_30,4,"[opening, overnight, premarket, regular]",-0.312848,0.312848,0.029710
4,60,bb_60_15,4,"[opening, overnight, premarket, regular]",-0.304833,0.304833,0.024496
5,60,bb_60_20,4,"[opening, overnight, premarket, regular]",-0.304833,0.304833,0.024496
6,60,rsi_14,4,"[opening, overnight, premarket, regular]",-0.302989,0.302989,0.017241
7,60,stoch_k_30,3,"[opening, premarket, regular]",-0.303724,0.303724,0.020031
8,60,bb_60_25,3,"[overnight, premarket, regular]",-0.255617,0.255617,0.020739
9,60,ema_20,2,"[opening, overnight]",-0.368128,0.368128,0.014816



3. Factores contextuales:



,horizon,indicator,regime,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
0,90,bb_30_20,[premarket],-0.210115,0.210115,0.011082
1,90,bb_30_25,[premarket],-0.210115,0.210115,0.011082



4. Top usados para el análisis:



,indicator,horizon,target_col,regime_flag,IC_IS,IC_OOS,oos_minus_is,n_pairs_IS,n_pairs_OOS,abs_IC_OOS,note,regime_name,same_sign,abs_gap
0,ema_60,60,delta_60,is_overnight,-0.394806,-0.370539,0.024267,107059,88486,0.370539,,overnight,True,0.024267
1,roc_60,60,delta_60,is_overnight,-0.372131,-0.359776,0.012356,107059,88486,0.359776,,overnight,True,0.012356
2,bb_60_15,60,delta_60,is_overnight,-0.324796,-0.310131,0.014664,107059,88486,0.310131,,overnight,True,0.014664
3,bb_60_20,60,delta_60,is_overnight,-0.324796,-0.310131,0.014664,107059,88486,0.310131,,overnight,True,0.014664
4,bb_60_25,60,delta_60,is_overnight,-0.324796,-0.310131,0.014664,107059,88486,0.310131,,overnight,True,0.014664
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,rsi_14,90,delta_90,is_regular,-0.265528,-0.256259,0.009269,170869,141226,0.256259,,regular,True,0.009269
76,ema_30,90,delta_90,is_regular,-0.265219,-0.254147,0.011072,170869,141226,0.254147,,regular,True,0.011072
77,roc_30,90,delta_90,is_regular,-0.252621,-0.242753,0.009869,170869,141226,0.242753,,regular,True,0.009869
78,stoch_k_30,90,delta_90,is_regular,-0.239019,-0.227848,0.011171,170869,141226,0.227848,,regular,True,0.011171


## **7.3. Observaciones**

**1. Diferencias claras por régimen**

* **Opening** es el régimen con mayor señal:

  * `mean_abs_IC_OOS` ≈ 0.38 – 0.40
    → Máxima predictibilidad

* **Overnight y premarket** tienen señal intermedia:

  * ≈ 0.19 – 0.21
    → Buena estructura antes de apertura

* **Regular** es el más débil:

  * ≈ 0.15 – 0.19
    → Mercado más eficiente

---

**2. Consistencia estructural**

* Los principales indicadores aparecen en **todos los regímenes (n_regimes = 4)**:

  * `ema_60`, `roc_60`, `ema_30`, `roc_30`, `bb_60_*`, `rsi_14`

* Mantienen:

  * alta magnitud OOS (~0.30 – 0.40)
  * bajo gap (~0.02)

→ Señal **estable, transversal al mercado**

---

**3. Factores parcialmente dependientes de régimen**

* Indicadores con `n_regimes = 2 o 3`:

  * `stoch_k_30`, `ema_20`, `roc_20`, `bb_60_25`

→ Funcionan bien, pero no en todos los contextos

---

**4. Factores contextuales (muy pocos)**

* Solo aparecen:

  * `bb_30_20`, `bb_30_25` en **premarket**

→ Señal específica de microestructura previa a apertura

---

**5. Alta estabilidad global**

* `pct_same_sign = 100%` en todos los casos
* `mean_abs_gap` bajo en todos los regímenes

→ No hay inestabilidad ni sobreajuste

## **7.4. Conclusiones parciales**


**1. Existen factores estructurales claros**

* `ema_60` y `roc_60` son los más fuertes
* Funcionan en todos los regímenes
  → **core features del modelo**

---

**2. La señal depende del régimen (en intensidad, no en dirección)**

* La dirección es la misma (mean reversion)
* Pero la **fuerza cambia significativamente**

→ El régimen es una variable relevante

---

**3. El modelo debería capturar contexto de mercado**

* Dado que:

  * opening ≫ overnight/premarket ≫ regular

→ Es recomendable incluir:

* flags de régimen (ya los tienes)
* o modelar explícitamente por régimen

---

**4. Los factores contextuales son marginales**

* Muy pocos indicadores son exclusivos de un régimen
  → La mayoría de la señal es **global**, no local

---

**Conclusión técnica del punto 7**

* El sistema presenta:

  * **factores estructurales dominantes**
  * **variación de intensidad por régimen**
  * **alta estabilidad entre contextos**

→ La predictibilidad no depende de descubrir nuevos indicadores,
sino de **cómo usar los mismos indicadores en distintos regímenes**

# **8. Ranking de indicadores**


En este punto se priorizan los indicadores técnicos en función de tres criterios clave:

* magnitud de la señal fuera de muestra (IC_OOS)
* estabilidad entre IS y OOS (gap bajo)
* consistencia de signo

El objetivo es identificar los factores más relevantes y confiables para el modelado.

---

### Criterios de ranking

Un indicador se considera de alta calidad cuando:

* presenta un **|IC_OOS| elevado** → señal fuerte
* tiene un **gap IS vs OOS bajo** → buena generalización
* mantiene el **mismo signo** → estabilidad direccional

La combinación de estos tres criterios permite filtrar indicadores robustos y evitar señales espurias.

---

### Resultados principales

**Indicadores mejor rankeados**

Los indicadores que dominan el ranking son:

* ema_60
* roc_60
* roc_30
* ema_30
* bb_60_*
* rsi_14

Estos presentan:

* alta magnitud de IC_OOS (~0.30 – 0.40)
* bajo gap (~0.02)
* consistencia de signo en todos los casos

---

**Dominancia de factores de tendencia y momentum**

Los indicadores mejor posicionados pertenecen principalmente a:

* medias móviles (EMA)
* rate of change (ROC)
* bandas de Bollinger
* RSI

Esto indica que la señal está dominada por dinámicas de:

* momentum suavizado
* reversión a la media

---

**Indicadores secundarios**

Otros indicadores relevantes, pero con menor consistencia o menor cobertura entre regímenes:

* stoch_k_*
* ema_20
* roc_20

Estos pueden aportar valor adicional, pero no constituyen el núcleo del sistema.

---

### Lectura estructural

* El ranking es consistente entre horizontes (delta_60 y delta_90)
* Los mismos indicadores aparecen en las primeras posiciones
* No hay dependencia fuerte de un único régimen

Esto refuerza la idea de que existen factores estructurales dominantes.

---

### Conclusión del ranking

* Se identifica un conjunto reducido de indicadores que concentran la mayor parte de la señal
* Estos indicadores son:

  * fuertes en OOS
  * estables
  * consistentes

→ Constituyen los **principales candidatos para feature selection**

---

Si quieres, el siguiente paso es dejar esto en una **lista final de features seleccionadas** para pasar al modelado.


## **8.1. Códigos de ranking**

### **1. Ranking global de indicadores**

In [84]:
ranking_df = (
    robustness_top_all
    .groupby("indicator")
    .agg(
        n_regimes=("regime_name", "nunique"),
        mean_IC_OOS=("IC_OOS", "mean"),
        mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
        mean_abs_gap=("abs_gap", "mean"),
        pct_same_sign=("same_sign", "mean"),
    )
    .reset_index()
)

ranking_df["pct_same_sign"] = ranking_df["pct_same_sign"] * 100

ranking_df = ranking_df.sort_values(
    ["mean_abs_IC_OOS", "n_regimes"],
    ascending=[False, False]
).reset_index(drop=True)



### **2. Top indicadores “core” (estructurales)**

In [85]:
core_indicators = ranking_df[
    ranking_df["n_regimes"] >= 4
].sort_values("mean_abs_IC_OOS", ascending=False)



### **3. Indicadores secundarios**

In [86]:
secondary_indicators = ranking_df[
    (ranking_df["n_regimes"] >= 2) &
    (ranking_df["n_regimes"] < 4)
].sort_values("mean_abs_IC_OOS", ascending=False)



### **4. Indicadores contextuales**

In [87]:
contextual_indicators = ranking_df[
    ranking_df["n_regimes"] == 1
].sort_values("mean_abs_IC_OOS", ascending=False)



### **5. Validación de estabilidad**

In [88]:
ranking_df[[
    "indicator",
    "mean_abs_IC_OOS",
    "mean_abs_gap",
    "pct_same_sign"
]].sort_values("mean_abs_IC_OOS", ascending=False).head(10)

,indicator,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign
0,ema_60,0.397697,0.022787,100.0
1,roc_60,0.392403,0.022463,100.0
2,stoch_k_20,0.373865,0.008610,100.0
3,roc_20,0.362270,0.019819,100.0
4,roc_30,0.335522,0.027379,100.0
5,ema_30,0.328344,0.019578,100.0
6,bb_60_15,0.326115,0.024568,100.0
7,ema_20,0.321288,0.018156,100.0
8,stoch_k_30,0.318178,0.013265,100.0
9,rsi_14,0.312140,0.016573,100.0


### **6. Clasificación por familia (para justificar momentum / mean reversion)**

In [89]:
def classify_family(indicator: str) -> str:
    s = indicator.lower()
    if s.startswith("ema_"): return "EMA"
    if s.startswith("roc_"): return "ROC"
    if s.startswith("rsi_"): return "RSI"
    if s.startswith("bb_"): return "BB"
    if s.startswith("stoch_"): return "STOCH"
    return "OTHER"

ranking_df["family"] = ranking_df["indicator"].apply(classify_family)

family_summary = (
    ranking_df.groupby("family")
    .agg(
        n_indicators=("indicator", "count"),
        mean_abs_IC_OOS=("mean_abs_IC_OOS", "mean")
    )
    .sort_values("mean_abs_IC_OOS", ascending=False)
)




## **8.2. Resultados**

In [92]:
print('\n Ranking global de indicadores: \n')
display (ranking_df.head(15))

print('\n Top indicadores “core” (estructurales): \n')
display (core_indicators)


print('\n Indicadores secundarios')
display(secondary_indicators)

print('\n Indicadores contextuales')
display(contextual_indicators)


print('\n Clasificación por familia (para justificar momentum / mean reversion)')
display(family_summary)


print('\nValidación de estabilidad')
ranking_df[[
    "indicator",
    "mean_abs_IC_OOS",
    "mean_abs_gap",
    "pct_same_sign"
]].sort_values("mean_abs_IC_OOS", ascending=False).head(10)



 Ranking global de indicadores: 



,indicator,n_regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign,family
0,ema_60,4,-0.397697,0.397697,0.022787,100.0,EMA
1,roc_60,4,-0.392403,0.392403,0.022463,100.0,ROC
2,stoch_k_20,2,-0.373865,0.373865,0.008610,100.0,STOCH
3,roc_20,2,-0.362270,0.362270,0.019819,100.0,ROC
4,roc_30,4,-0.335522,0.335522,0.027379,100.0,ROC
5,ema_30,4,-0.328344,0.328344,0.019578,100.0,EMA
6,bb_60_15,4,-0.326115,0.326115,0.024568,100.0,BB
7,ema_20,3,-0.321288,0.321288,0.018156,100.0,EMA
8,stoch_k_30,3,-0.318178,0.318178,0.013265,100.0,STOCH
9,rsi_14,4,-0.312140,0.312140,0.016573,100.0,RSI



 Top indicadores “core” (estructurales): 



,indicator,n_regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign
0,ema_60,4,-0.397697,0.397697,0.022787,100.0
1,roc_60,4,-0.392403,0.392403,0.022463,100.0
4,roc_30,4,-0.335522,0.335522,0.027379,100.0
5,ema_30,4,-0.328344,0.328344,0.019578,100.0
6,bb_60_15,4,-0.326115,0.326115,0.024568,100.0
9,rsi_14,4,-0.312140,0.312140,0.016573,100.0



 Indicadores secundarios


...

,indicator,n_regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign
12,bb_30_20,1,-0.210115,0.210115,0.011082,100.0
13,bb_30_25,1,-0.210115,0.210115,0.011082,100.0



 Clasificación por familia (para justificar momentum / mean reversion)


,n_indicators,mean_abs_IC_OOS
family,,
ROC,3,0.363398
EMA,3,0.349109
STOCH,2,0.346021
RSI,1,0.312140
BB,6,0.247056



Validación de estabilidad


,indicator,mean_abs_IC_OOS,mean_abs_gap,pct_same_sign
0,ema_60,0.397697,0.022787,100.0
1,roc_60,0.392403,0.022463,100.0
2,stoch_k_20,0.373865,0.008610,100.0
3,roc_20,0.362270,0.019819,100.0
4,roc_30,0.335522,0.027379,100.0
5,ema_30,0.328344,0.019578,100.0
6,bb_60_15,0.326115,0.024568,100.0
7,ema_20,0.321288,0.018156,100.0
8,stoch_k_30,0.318178,0.013265,100.0
9,rsi_14,0.312140,0.016573,100.0


## **8.3. Observaciones**

**1. Dominancia clara de ciertos indicadores**

* Los indicadores mejor rankeados son:

  * `ema_60`
  * `roc_60`

* Presentan:

  * **máxima magnitud OOS (~0.39 – 0.40)**
  * **gap bajo (~0.02)**
  * **100% consistencia de signo**

→ Son los factores más fuertes del sistema

---

**2. Existencia de un núcleo estructural sólido**

Los indicadores “core” (n_regimes = 4):

* ema_60
* roc_60
* roc_30
* ema_30
* bb_60_15
* rsi_14

Comparten:

* alta señal (≥ 0.30)
* estabilidad
* presencia en todos los regímenes

→ Constituyen el **núcleo del modelo**

---

**3. Indicadores secundarios con alta señal**

Algunos indicadores no estructurales presentan alta magnitud:

* `stoch_k_20` (~0.37)
* `roc_20` (~0.36)

Pero:

* aparecen en pocos regímenes
  → Son **potentes pero contextuales**

---

**4. Indicadores contextuales claramente identificados**

* `bb_30_20`, `bb_30_25` (solo premarket)

→ Señales específicas de microestructura
→ No generalizan

---

**5. Estabilidad total del sistema**

* `pct_same_sign = 100%` en todos los indicadores
* `mean_abs_gap` bajo en todos los casos (~0.01 – 0.03)

→ No hay evidencia de inestabilidad ni overfitting

---

**6. Dominancia por familia**

Ranking por familias:

1. ROC → mayor señal (~0.36)
2. EMA → muy fuerte (~0.35)
3. STOCH → alto pero menos general
4. RSI → consistente
5. BB → más débil en promedio

→ La señal está dominada por:

* **momentum (ROC)**
* **tendencia suavizada (EMA)**


## **8.4. Conclusiones del ranking**

**1. Existe un conjunto reducido de indicadores dominantes**

* La mayor parte de la señal se concentra en pocos indicadores
  → No es necesario un gran número de features

---

**2. Los factores principales son estructurales**

* EMA y ROC dominan en todos los regímenes
  → Son **robustos y generalizables**

---

**3. La señal es consistente y confiable**

* Alta magnitud OOS
* Bajo gap
* 100% consistencia de signo

→ Cumple todos los criterios de calidad

---

**4. Los indicadores secundarios pueden aportar valor incremental**

* Especialmente:

  * STOCH
  * ROC corto
  * EMA corto

→ útiles como features complementarias




## **8.5. Conclusión técnica del punto 8**


* El ranking confirma que:

  * la señal está concentrada en pocos factores
  * estos factores son estables y robustos
  * la estructura del problema está bien definida

→ Ya es posible definir un **subset óptimo de features** para modelado

# **9. Análisis de repetición entre regímenes**
   



   Se identifican indicadores que:

* aparecen consistentemente en varios regímenes

Objetivo:
Detectar factores robustos globales.

## **9.1. Código**

### **1. Análisis de repetición entre regímenes**

In [98]:
# ============================================================
# 1) FRECUENCIA DE APARICIÓN POR RÉGIMEN
# ============================================================

repetition_df = (
    robustness_top_all
    .groupby("indicator")
    .agg(
        n_regimes=("regime_name", "nunique"),
        regimes=("regime_name", lambda x: sorted(x.unique())),
        mean_IC_OOS=("IC_OOS", "mean"),
        mean_abs_IC_OOS=("abs_IC_OOS", "mean"),
        mean_abs_gap=("abs_gap", "mean"),
    )
    .reset_index()
    .sort_values(["n_regimes", "mean_abs_IC_OOS"], ascending=[False, False])
)



### **2. Indicadores globales**

In [100]:
global_factors = repetition_df[
    repetition_df["n_regimes"] >= 4
]


### **3. Indicadores semi-globales**

In [101]:
semi_global_factors = repetition_df[
    (repetition_df["n_regimes"] == 3)
]


### **4. Indicadores débiles / locales**

In [102]:
local_factors = repetition_df[
    repetition_df["n_regimes"] <= 2
]


### **5. Validación: relación repetición vs señal**

In [97]:
repetition_df[[
    "indicator",
    "n_regimes",
    "mean_abs_IC_OOS",
    "mean_abs_gap"
]].sort_values(["n_regimes", "mean_abs_IC_OOS"], ascending=[False, False])

,indicator,n_regimes,mean_abs_IC_OOS,mean_abs_gap
8,ema_60,4,0.397697,0.022787
11,roc_60,4,0.392403,0.022463
10,roc_30,4,0.335522,0.027379
7,ema_30,4,0.328344,0.019578
3,bb_60_15,4,0.326115,0.024568
12,rsi_14,4,0.312140,0.016573
6,ema_20,3,0.321288,0.018156
14,stoch_k_30,3,0.318178,0.013265
4,bb_60_20,3,0.269376,0.020604
5,bb_60_25,3,0.269376,0.020604


## **9.2. Resultados**

In [103]:

print('\n1. Análisis de repetición entre regímenes')
display(repetition_df)

print('\n2. Indicadores globales')
display(global_factors)

print('\n3. Indicadores semi-globales')
display(semi_global_factors)

print('\n4. Indicadores débiles / locales')
display(local_factors)

print('\n5. Validación: relación repetición vs señal')
repetition_df[[
    "indicator",
    "n_regimes",
    "mean_abs_IC_OOS",
    "mean_abs_gap"
]].sort_values(["n_regimes", "mean_abs_IC_OOS"], ascending=[False, False])


1. Análisis de repetición entre regímenes


,indicator,n_regimes,regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
8,ema_60,4,"[opening, overnight, premarket, regular]",-0.397697,0.397697,0.022787
11,roc_60,4,"[opening, overnight, premarket, regular]",-0.392403,0.392403,0.022463
10,roc_30,4,"[opening, overnight, premarket, regular]",-0.335522,0.335522,0.027379
7,ema_30,4,"[opening, overnight, premarket, regular]",-0.328344,0.328344,0.019578
3,bb_60_15,4,"[opening, overnight, premarket, regular]",-0.326115,0.326115,0.024568
12,rsi_14,4,"[opening, overnight, premarket, regular]",-0.312140,0.312140,0.016573
6,ema_20,3,"[opening, overnight, premarket]",-0.321288,0.321288,0.018156
14,stoch_k_30,3,"[opening, premarket, regular]",-0.318178,0.318178,0.013265
4,bb_60_20,3,"[overnight, premarket, regular]",-0.269376,0.269376,0.020604
5,bb_60_25,3,"[overnight, premarket, regular]",-0.269376,0.269376,0.020604



2. Indicadores globales


,indicator,n_regimes,regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
8,ema_60,4,"[opening, overnight, premarket, regular]",-0.397697,0.397697,0.022787
11,roc_60,4,"[opening, overnight, premarket, regular]",-0.392403,0.392403,0.022463
10,roc_30,4,"[opening, overnight, premarket, regular]",-0.335522,0.335522,0.027379
7,ema_30,4,"[opening, overnight, premarket, regular]",-0.328344,0.328344,0.019578
3,bb_60_15,4,"[opening, overnight, premarket, regular]",-0.326115,0.326115,0.024568
12,rsi_14,4,"[opening, overnight, premarket, regular]",-0.312140,0.312140,0.016573



3. Indicadores semi-globales


,indicator,n_regimes,regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
6,ema_20,3,"[opening, overnight, premarket]",-0.321288,0.321288,0.018156
14,stoch_k_30,3,"[opening, premarket, regular]",-0.318178,0.318178,0.013265
4,bb_60_20,3,"[overnight, premarket, regular]",-0.269376,0.269376,0.020604
5,bb_60_25,3,"[overnight, premarket, regular]",-0.269376,0.269376,0.020604



4. Indicadores débiles / locales


,indicator,n_regimes,regimes,mean_IC_OOS,mean_abs_IC_OOS,mean_abs_gap
13,stoch_k_20,2,"[opening, premarket]",-0.373865,0.373865,0.008610
9,roc_20,2,"[opening, overnight]",-0.362270,0.362270,0.019819
0,bb_30_15,2,"[premarket, regular]",-0.197239,0.197239,0.009527
1,bb_30_20,1,[premarket],-0.210115,0.210115,0.011082
2,bb_30_25,1,[premarket],-0.210115,0.210115,0.011082



5. Validación: relación repetición vs señal


,indicator,n_regimes,mean_abs_IC_OOS,mean_abs_gap
8,ema_60,4,0.397697,0.022787
11,roc_60,4,0.392403,0.022463
10,roc_30,4,0.335522,0.027379
7,ema_30,4,0.328344,0.019578
3,bb_60_15,4,0.326115,0.024568
12,rsi_14,4,0.312140,0.016573
6,ema_20,3,0.321288,0.018156
14,stoch_k_30,3,0.318178,0.013265
4,bb_60_20,3,0.269376,0.020604
5,bb_60_25,3,0.269376,0.020604


## **9.3. Análisis de resultados**


**1. Existen indicadores claramente globales**

* Los más importantes son:

  * `ema_60`, `roc_60`
  * `ema_30`, `roc_30`
  * `bb_60_15`, `rsi_14`

* Aparecen en **todos los regímenes**
  → Son **robustos y confiables**

---

**2. La señal principal es estructural**

* Los indicadores que más se repiten:

  * también son los que tienen mayor IC

→ La señal **no depende del régimen**, es general del mercado

---

**3. Más repetición = mejor indicador**

* A mayor `n_regimes`:

  * mayor consistencia
  * mayor estabilidad
  * mejor desempeño

→ La repetición es un **criterio clave de calidad**

---

**4. Existen indicadores útiles pero no globales**

* Ejemplo:

  * `ema_20`, `stoch_k_30`, `bb_60_20`

* Funcionan bien, pero:

  * no en todos los regímenes

→ Son **complementarios**, no principales

---

**5. Algunos indicadores son solo contextuales**

* Ejemplo:

  * `bb_30_20`, `bb_30_25` (solo premarket)

→ No generalizan
→ Dependen del contexto

---

**6. Importante: señal alta no implica robustez**

* Ejemplo:

  * `stoch_k_20`, `roc_20` tienen IC alto

Pero:

* aparecen en pocos regímenes

→ Son **fuertes pero poco robustos**

---

**Conclusión final del punto 9**

* Los mejores indicadores son:

  * **los que se repiten en todos los regímenes**
  * **no los que tienen solo IC alto**

→ La señal del problema es **global y estructural**, no específica de un régimen


# **10. Identificación de indicadores específicos por régimen**

Se buscan factores que:

* funcionan bien en un régimen
* no en otros

Objetivo:
Capturar señales contextuales.

## **10.1. Código**

In [109]:
rows = []

for regime, s in sets_by_regime.items():
    others = set().union(*[v for k, v in sets_by_regime.items() if k != regime])
    specific = s - others

    df_reg = top_by_regime[top_by_regime["regime_name"] == regime]

    for ind in specific:
        row = df_reg[df_reg["indicator"] == ind].iloc[0]
        rows.append({
            "regime": regime,
            "indicator": ind,
            "IC_OOS": row["IC_OOS"],
            "abs_IC_OOS": row["abs_IC_OOS"],
            "abs_gap": row["abs_gap"],
        })

# 👇 manejo del caso vacío
if len(rows) == 0:
    specific_by_regime = pd.DataFrame(
        columns=["regime", "indicator", "IC_OOS", "abs_IC_OOS", "abs_gap"]
    )
else:
    specific_by_regime = (
        pd.DataFrame(rows)
        .sort_values(["regime", "abs_IC_OOS"], ascending=[True, False])
    )

specific_by_regime

,regime,indicator,IC_OOS,abs_IC_OOS,abs_gap


##**10.2. Resultados**

## **10.3. Observaciones**

# **11. Análisis de redundancia (correlación entre indicadores)**

Se evalúa:

* correlación entre features
* agrupación de indicadores similares

Objetivo:

* eliminar duplicados
* reducir dimensionalidad
* evitar multicolinealidad


# **12. Selección final de features**

Se seleccionan indicadores según:

* señal OOS
* robustez
* consistencia
* baja redundancia

Objetivo:
Definir el feature set final.

# **13. Interpretación económica de los factores**

Se analiza qué tipo de señal capturan:

* momentum
* mean reversion
* volatilidad

Objetivo:
Validar coherencia con el mercado.

# **14. Output del stage**

Se generan:

* tablas resumen
* rankings
* análisis por régimen
* features finales
* conclusiones

Objetivo final:
Determinar si existe señal predictiva y definir un conjunto robusto de features para el modelado.